In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

from openai import api_key
import json
import time 
import re 
import unicodedata
from datetime import datetime
from typing import TypedDict, List, Dict, Any, Optional

from langgraph.graph import StateGraph, END,START
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers.pydantic import PydanticOutputParser
from neo4j import GraphDatabase
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
import subprocess
from langchain_mistralai import ChatMistralAI
subprocess.run(["pip", "install", "reportlab"], capture_output=True)

# ─────────────────────────────────────────────────────────────────
# CONFIG – Load all secrets from environment variables
# ─────────────────────────────────────────────────────────────────
NEO4J_URI      = os.getenv("NEO4J_URI", "neo4j://127.0.0.1:7687")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
if not NEO4J_PASSWORD:
    raise ValueError("Missing NEO4J_PASSWORD in .env file")      
BATCH_SIZE     = 10   # triples per LLM call in detect_hallucinations

# Neo4j driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Groq (Llama)
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("Missing GROQ_API_KEY in .env file")
llm_reasoning_llama = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0,
    api_key=groq_api_key,
)

# Mistral AI
mistral_api_key = os.getenv("MISTRAL_API_KEY")
if not mistral_api_key:
    raise ValueError("Missing MISTRAL_API_KEY in .env file")
llm_reasoning_mistral = ChatMistralAI(
    model="mistral-large-latest",
    
    temperature=0
)

# Ollama (GLM on cloud) – note: requires a bearer token if using cloud
ollama_bearer_token = os.getenv("OLLAMA_BEARER_TOKEN", "")
llm_reasoning_glm = ChatOllama(
    model="glm-4.7:cloud",
    base_url="https://api.ollama.com",  # correct cloud API endpoint
    client_kwargs={
        "headers": {
            "Authorization": "Bearer "
        }
    }
)

# OpenAI (GitHub Marketplace inference)
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("Missing OPENAI_API_KEY in .env file")
llm_reasoning_gpt = ChatOpenAI(
    model="openai/gpt-4.1",  
    api_key="",
    base_url="https://models.github.ai/inference"
)



# Gemini
gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("Missing GEMINI_API_KEY in .env file")
llm_reasoning_gemini = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  
    api_key=gemini_api_key,
)

In [42]:
# ─────────────────────────────────────────────────────────────────
# STATE  (plain dict — replaces LangGraph TypedDict/StateGraph)
# ─────────────────────────────────────────────────────────────────
# We keep the TypedDict as a documentation aid; it is NOT used by
# any framework — just passed as a regular dict between steps.
class PipelineState(TypedDict):
    user_input:             str
    raw_response:           str
    extracted_triples:      List[Dict[str, str]]     
    canonicalized_triples:  List[Dict[str, Any]]
    canonicalized_triples_Before:  List[Dict[str, Any]] 
    symbolic_violations:    List[Dict[str, Any]]
    neo4j_evidence:         List[Dict[str, Any]]
    hallucination_report:   Any
    logical_inconsistencies:List[Dict[str, Any]]

In [43]:
# ─────────────────────────────────────────────────────────────────
# UTILITIES  (unchanged from original)
# ─────────────────────────────────────────────────────────────────
def _strip_fences(text: str) -> str:
    """Remove markdown code fences from LLM output."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$",          "", text)
    return text.strip()


def _safe_parse_list(raw: str, label: str) -> list:
    try:
        result = json.loads(raw)
        return result if isinstance(result, list) else []
    except (json.JSONDecodeError, ValueError) as e:
        print(f"[{label}] JSON parse error: {e} — attempting truncation recovery")
        # NEW: show the part near the error position
        if hasattr(e, 'pos') and e.pos:
            start = max(0, e.pos - 50)
            end = min(len(raw), e.pos + 50)
            print(f"[{label}] Error context: ...{raw[start:end]}...")
        # Your existing recovery logic...
        last_valid = raw.rfind("\n  }")
        if last_valid != -1:
            recovered = raw[: last_valid + 4] + "\n]"
            try:
                result = json.loads(recovered)
                result = result if isinstance(result, list) else []
                print(f"[{label}] recovered {len(result)} items")
                return result
            except (json.JSONDecodeError, ValueError):
                pass
        print(f"[{label}] recovery failed — returning []")
        # NEW: log the tail of the raw response to see if it's cut off
        print(f"[{label}] Raw response tail: ...{raw[-200:]}")
        return []

In [44]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal

from pydantic import BaseModel, Field, model_validator
from typing import Literal
import json

class VerifiableTriple(BaseModel):
    subject: str
    predicate: str
    object: str
    predicate_definition: str = Field(
        default="",                      # ← empty string if LLM omits it
        description="One sentence defining what this relationship means"
    )

class KnowledgeGraph(BaseModel):
    triples: list[VerifiableTriple]

    @model_validator(mode="before")
    @classmethod
    def parse_triples_if_string(cls, values):
        triples = values.get("triples")
        if isinstance(triples, str):
            try:
                parsed = json.loads(triples)
                values["triples"] = parsed if isinstance(parsed, list) else []
            except (json.JSONDecodeError, ValueError):
                values["triples"] = []
        return values


# --- Stage 1: Answer prompt ---
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a knowledgeable assistant. Answer the user's question clearly and comprehensively using the given context."),
    ("user", "{user_input}")
])


# --- Stage 2: Extract triples prompt (Optimized for Extrapolation & Recall) ---
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert at extracting information in structured formats to build a knowledge graph.
Think step-by-step through two stages internally, then output only the final JSON array.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STAGE 1 — ENTITY EXTRACTION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Scan the text for all unique entities (Nodes). 


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STAGE 2 — RELATION EXTRACTION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Identify all relationships (Edges) between the entities found in Stage 1.
2. Predicates must be specific, directed, and written in UPPERCASE (e.g., HAS_INTEREST_RATE, HAS_RISK_TIER).
3. Ensure every triple follows the structure: [Subject] -> [Predicate] -> [Object].
4. **INCLUSION RULE**: Extract both explicit facts and logical extrapolations/estimates provided in the text and range and below and above values.Also extract ones with objects semantically saying not included 
   or not provided or unkown or not in context 
5. **VERIFICATION**: A triple is valid if the author asserts it, even if labeled as "extrapolated" or "estimated."
6- **Exclusion RULE** Any calculation steps should not be included.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1-The subject entity hould include only the number 
e.g: WRITTEN WORDS 300 -> 300
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT FORMAT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Return exactly this JSON structure:
{{
  "subject":              "<entity name>",
  "predicate":            "<UPPER_SNAKE_CASE relationship type>",
  "object":               "<cleaned value or entity name>",
  "predicate_definition": "<one sentence defining what this relationship means>"
}}

Output ONLY the JSON array. No markdown fences, no explanation.
"""),
    ("user", "LLM Answer to extract triples from:\n\n{answer}")
])


structured_extractor = extract_prompt | llm_reasoning_gpt
#structured_extractor5 = extract_prompt | llm_reasoning_gemini.with_structured_output(KnowledgeGraph)

def extract_triples(state: dict) -> dict:
    # Stage 1: LLM answers the question
    answer_chain = answer_prompt | llm_reasoning_llama
    answer = answer_chain.invoke({"user_input": state["user_input"]})
    generated_answer = answer.content
    print(generated_answer)
    state["raw_response"] = generated_answer

    # ── helper: safe invoke for models with with_structured_output ──
    def safe_invoke(extractor, label):
        try:
            return extractor.invoke({"answer": generated_answer})
        except Exception as e:
            print(f"{label} extractor failed: {e}")
            return KnowledgeGraph(triples=[])

    # ── helper: parse GLM raw AIMessage manually ──
    def parse_glm(generated_answer: str) -> KnowledgeGraph:
        try:
            glm_response = structured_extractor.invoke({"answer": generated_answer})
            glm_content = glm_response.content
            if "```json" in glm_content:
                glm_content = glm_content.split("```json")[1].split("```")[0]
            elif "```" in glm_content:
                glm_content = glm_content.split("```")[1].split("```")[0]
            glm_data = json.loads(glm_content.strip())
            if isinstance(glm_data, list):
                return KnowledgeGraph(triples=glm_data)
            elif isinstance(glm_data, dict) and "triples" in glm_data:
                return KnowledgeGraph(triples=glm_data["triples"])
            elif isinstance(glm_data, dict) and "subject" in glm_data:
                return KnowledgeGraph(triples=[glm_data])
            else:
                return KnowledgeGraph(triples=[])
        except Exception as e:
            print(f"GLM extractor failed: {e}")
            return KnowledgeGraph(triples=[])

    # Stage 2: Extract verifiable triples from the answer
    

    kg = parse_glm(generated_answer)
    #kg5 = safe_invoke(structured_extractor5, "Gemini")

    all_kgs = [
        (kg, "GLM"),
        #(kg5, "Gemini"),
    ]
    verifiable = [t.model_dump() for t in kg.triples ]
    print("\nExtracted triples (JSON):")
    print(json.dumps(verifiable, indent=2))
    state["extracted_triples"] = verifiable
    return state

In [45]:
# STEP 3 — canonicalize_triples
# ─────────────────────────────────────────────────────────────────
CANONICALIZE_SYSTEM_PROMPT = """...
You will receive:
- "relationship_types": the full list of DB relationship types (shared for all triples)
- "triples": list of triples to normalize, each with "triple" and "predicate_definition"

Match each triple's predicate against the shared relationship_types list.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SUBJECT & object NORMALIZATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
-Always extract the number only from the subject for credit scores 
any additional text written with it remove it and return only the number for the subject 
e.g: "Subject":"Words written with the credit score 300" -> "Subject":"300"

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PREDICATE NORMALIZATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Use the "relationship_types" list for THIS triple only.

STEP 1 — Strip qualifying adjectives before matching.
  Extracted predicates often carry adjectives that narrow meaning but have
  no dedicated DB relationship. Strip them first, then match the core verb phrase.

  Common qualifiers to strip (not exhaustive — apply this logic to any domain):
    Certainty      : known, estimated, approximate, confirmed, expected
    Temporality    : current, previous, initial, final, latest
    Ordinality     : minimum, maximum, total, average
    Source         : calculated, derived, assigned, reported

  Examples:
    "HAS_KNOWN_VALUE"      → core: "HAS_VALUE"
    "HAS_ESTIMATED_VALUE"  → core: "HAS_VALUE"
    "HAS_MINIMUM_SCORE"    → core: "HAS_SCORE"
    "HAS_FINAL_STATUS"     → core: "HAS_STATUS"
    "REQUIRES_MINIMUM_X"   → core: "REQUIRES_X"
    "HAS_INTERPOLATED_VALUE"      → core: "HAS_VALUE"

STEP 2 — Match the core predicate. Try in order:
  1. Exact match (case-insensitive, ignoring underscores vs spaces)
  2. Qualifier-stripped match (from Step 1 above)
  3. Synonym or natural-language paraphrase
       e.g. "needs" → "REQUIRES",  "must pass" → "REQUIRES_PASSING"
  4. Partial overlap — the list entry shares the CORE VERB of the extracted
     predicate AND belongs to the same semantic domain

STEP 3 — Preserve qualifier meaning in the object when stripping.
  If you stripped a qualifier to achieve a match, check whether the object
  already captures that distinction. If not, append it as a parenthetical
  annotation to preserve the original meaning for downstream processing.

  Examples:
    predicate "HAS_ESTIMATED_VALUE"  matched to "HAS_VALUE"
    object "42.5"  →  "42.5 (estimated)"      ← qualifier moved to object

    predicate "HAS_KNOWN_RATE"  matched to "HAS_RATE"
    object "5.2%"  →  "5.2% (known)"          ← qualifier moved to object

    predicate "HAS_MINIMUM_SCORE"  matched to "HAS_SCORE"
    object "passing grade"  →  already descriptive, no annotation needed

⚠ REJECTION RULE — Do NOT match, set predicate to ALL_CAPS_WITH_UNDERSCORES
of the original, and set confidence "LOW" if ANY of these are true:
  • After qualifier stripping, no relationship type shares the core verb
    AND semantic domain.
  • The match requires ignoring more than one CONTENT word of the core
    predicate (after qualifier stripping).
  • The only shared words are generic stop-verb terms (e.g. "has", "is")
    while the specific meaning differs entirely.
  • The extracted predicate describes a mathematical or computational
    operation (e.g. "interpolated between", "calculated from", "derived as")
    and no DB relationship type encodes that same operation.

CONFIDENCE AFTER QUALIFIER STRIPPING:
  Exact or near-exact match (no stripping needed)    → HIGH
  Matched after stripping ONE qualifier              → MEDIUM
  Matched after stripping + synonym inference        → MEDIUM
  No match after all steps / REJECTION RULE hit      → LOW

Do NOT force a match to avoid LOW confidence. A correct LOW is more useful
than a wrong HIGH or MEDIUM.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CONFIDENCE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
HIGH   — predicate matched exactly or near-exactly.
MEDIUM — predicate matched with inference (synonym/paraphrase/qualifier-stripping).
         ALSO use MEDIUM when: the predicate IS matched to a known DB relationship
         type.
LOW    — predicate can not be matched to any db schema predicate

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT FORMAT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Return a JSON array, one object per triple:
{
  "original":         { verbatim extracted triple, unchanged },
  "canonical":        {
                        "subject":   "<The canonicalized subject>",
                        "predicate": "<matched DB relationship type>",
                        "object":    "<copied unchanged from original>"
                      },
  "match_confidence": "HIGH" | "MEDIUM" | "LOW",   ← must be this exact key
  "reasoning":        "one sentence covering the predicate match only"
}

Output ONLY valid JSON, no markdown fences."""


# Global cache for relationship types (fetch once)
_global_rel_types = None

def _get_global_relationship_types():
    global _global_rel_types
    if _global_rel_types is None:
        with driver.session() as session:
            result = session.run("MATCH ()-[r]->() RETURN DISTINCT type(r) AS rel_type")
            _global_rel_types = [r["rel_type"] for r in result]
    return _global_rel_types


from typing import Literal  # add this import at top if not present

class CanonicalizedTriple(BaseModel):
    subject: str = Field(description="Subject entity name, passed through unchanged")
    predicate: str = Field(description="Original predicate from extraction")
    object: str = Field(description="Object value, passed through unchanged")
    canonicalized_predicate: str = Field(
        default="",                      # ← default: empty if LLM omits it
        description="Matched database relationship type (or original if no match)"
    )
    match_confidence: Literal["HIGH", "MEDIUM", "LOW"] = Field(
        default="LOW",                   # ← default: LOW if LLM omits it
        description="Confidence level: HIGH, MEDIUM, or LOW"
    )
    matched_node_info: Optional[str] = Field(
        None, description="Brief description of matched node (optional)"
    )

    @model_validator(mode="before")
    @classmethod
    def fill_missing_canonicalized(cls, values):
        """
        If LLM returns the output format with 'canonical.predicate' instead of
        'canonicalized_predicate', remap it. Also fill defaults from original fields.
        """
        # Handle case where LLM returns nested canonical dict instead of flat fields
        if "canonical" in values and isinstance(values["canonical"], dict):
            canon = values["canonical"]
            if not values.get("canonicalized_predicate"):
                values["canonicalized_predicate"] = canon.get("predicate", "")
            if not values.get("subject"):
                values["subject"] = canon.get("subject", values.get("subject", ""))
            if not values.get("object"):
                values["object"] = canon.get("object", values.get("object", ""))

        # Fallback: if canonicalized_predicate still empty, copy from predicate
        if not values.get("canonicalized_predicate"):
            values["canonicalized_predicate"] = values.get("predicate", "")

        return values


class CanonicalizedOutput(BaseModel):
    triples: List[CanonicalizedTriple]

    @model_validator(mode="before")
    @classmethod
    def parse_triples_if_string(cls, values):
        triples = values.get("triples")
        if isinstance(triples, str):
            try:
                parsed = json.loads(triples)
                values["triples"] = parsed if isinstance(parsed, list) else []
            except (json.JSONDecodeError, ValueError):
                values["triples"] = []
        return values

# Instantiate the parser once (outside the function)
parser = PydanticOutputParser(pydantic_object=CanonicalizedOutput)

# ── correct schema ──

canonicalize_chain = llm_reasoning_gpt.with_structured_output(CanonicalizedOutput)
#canonicalize_chain = llm_reasoning_glm

def canonicalize_triples(state: dict) -> dict:
    extracted = state.get("extracted_triples", [])
    if not extracted:
        state["canonicalized_triples"] = []
        return state

    global_rel_types = _get_global_relationship_types()

    human_payload = {
        "relationship_types": global_rel_types,
        "triples": [
            {
                "subject":              triple.get("subject",              ""),
                "predicate":            triple.get("predicate",            ""),
                "object":               triple.get("object",               ""),
                "predicate_definition": triple.get("predicate_definition", ""),
            }
            for triple in extracted
        ],
    }

    messages = [
        SystemMessage(content=CANONICALIZE_SYSTEM_PROMPT),
        HumanMessage(content=json.dumps(human_payload, separators=(',', ':'))),
    ]

    canonicalized = []
    try:
        response = canonicalize_chain.invoke(messages)

        # ── GPT returns Pydantic object directly (has .triples) ──
        if hasattr(response, "triples"):
            for t, triple in zip(response.triples, extracted):
                canonicalized.append({
                    "original": {
                        "subject":   triple.get("subject",   ""),
                        "predicate": triple.get("predicate", ""),
                        "object":    triple.get("object",    ""),
                    },
                    "canonical": {
                        "subject":   t.subject,
                        "predicate": t.canonicalized_predicate or t.predicate,
                        "object":    t.object,
                    },
                    "match_confidence": t.match_confidence,
                })
            print(f"[canonicalize_triples] GPT parsed {len(canonicalized)} triples")

        # ── GLM/raw models return AIMessage (has .content) ──
        elif hasattr(response, "content"):
            raw_text = _strip_fences(response.content)
            raw_list = _safe_parse_list(raw_text, "canonicalize_triples")

            for item, triple in zip(raw_list, extracted):
                if not isinstance(item, dict):
                    continue
                original = item.get("original", {})
                canonical = item.get("canonical", {})

                if isinstance(original, dict) and original:
                    canonicalized.append({
                        "original": {
                            "subject":   original.get("subject",   triple.get("subject",   "")),
                            "predicate": original.get("predicate", triple.get("predicate", "")),
                            "object":    original.get("object",    triple.get("object",    "")),
                        },
                        "canonical": {
                            "subject":   canonical.get("subject",   triple.get("subject",   "")),
                            "predicate": canonical.get("predicate", triple.get("predicate", "")),
                            "object":    canonical.get("object",    triple.get("object",    "")),
                        },
                        "match_confidence": item.get("match_confidence", "LOW"),
                    })
                else:
                    canonicalized.append({
                        "original": {
                            "subject":   triple.get("subject",   ""),
                            "predicate": triple.get("predicate", ""),
                            "object":    triple.get("object",    ""),
                        },
                        "canonical": {
                            "subject":   triple.get("subject",   ""),
                            "predicate": item.get("canonicalized_predicate", triple.get("predicate", "")),
                            "object":    triple.get("object",    ""),
                        },
                        "match_confidence": item.get("match_confidence", "LOW"),
                    })
            print(f"[canonicalize_triples] raw model parsed {len(canonicalized)} triples")

        else:
            print(f"[canonicalize_triples] unexpected response type: {type(response)}")

    except Exception as e:
        print(f"[canonicalize_triples] failed: {e}")
        canonicalized = []

    
    state["canonicalized_triples"]        = canonicalized
    state["canonicalized_triples_Before"] = canonicalized
    print(f"[canonicalize_triples] canonicalized {len(canonicalized)} triples")
    return state

In [46]:
# ─────────────────────────────────────────────────────────────────
# Domain rules
# ─────────────────────────────────────────────────────────────────

MONOTONIC_DECREASING = {   # must DECREASE as credit score increases
    "INTEREST_RATE_CHARGED_AS_ANNUAL_BORROWING_COST_PERCENTAGE",
    "DEFAULT_PROBABILITY_RECORDED_AS_HISTORICAL_NONREPAYMENT_RATE",
    "DOWN_PAYMENT_REQUIRED_AS_UPFRONT_BORROWER_CASH_CONTRIBUTION",
}

MONOTONIC_INCREASING = {   # must INCREASE as credit score increases
    "APPROVAL_ODDS_ESTIMATED_AS_LENDER_ACCEPTANCE_LIKELIHOOD",
    "CREDIT_LIMIT_MULTIPLIER_APPLIED_AS_INCOME_BASED_BORROWING_CEILING_FACTOR",
    "MAX_DTI_PERMITTED_AS_MONTHLY_DEBT_BURDEN_CEILING",
}

TIER_ORDER = {
    "deep subprime": 1,
    "subprime":      2,
    "near prime":    3,
    "prime":         4,
    "super prime":   5,
}

LOGICAL_INCONSISTENCY_SYSTEM_PROMPT = """
You are a logical consistency checker for credit score knowledge graph triples.
Your ONLY job is to identify logical contradictions that exist WITHIN the provided
triples themselves — no external database, no domain facts, no outside knowledge.

A logical inconsistency means two or more triples directly contradict each other
by their own content, like saying something is both hot and cold at the same time.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INPUT STRUCTURE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
You will receive three pre-grouped sets of triples:

1. "same_subject_same_predicate"
   Triples sharing the same subject AND predicate but with different objects.
   Check if any two objects directly contradict each other.
   Type: DIRECT_CONFLICT

2. "same_predicate_different_scores"
   Triples sharing the same predicate but from different credit score subjects.
   Each predicate has a required direction:
     - DECREASING: higher credit score must NOT have a strictly HIGHER numeric value
    (equal values are acceptable and must NOT be flagged)
    - INCREASING: higher credit score must NOT have a strictly LOWER numeric value  
    (equal values are acceptable and must NOT be flagged)
     - TIER: higher credit score must have a higher or equal tier rank
       (deep subprime < subprime < near prime < prime < super prime)
   Extract the numeric score from the subject and the numeric value from the object,
   then check if any pair violates the required direction.
   Type: MONOTONICITY_VIOLATION or TIER_ORDER_VIOLATION

3. "all_triples"
   Complete flat list for reference.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INCONSISTENCY TYPES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

A. DIRECT_CONFLICT
   Same subject and predicate but two different objects simultaneously.
   Example: (600, HAS_INTEREST_RATE, 8%) AND (600, HAS_INTEREST_RATE, 15%)

B. MONOTONICITY_VIOLATION
   Higher credit score has a worse value than a lower credit score
   for a predicate that must strictly improve with score.
   Example: score 700 has interest rate 9.5% but score 580 has interest rate 7.2%

C. TIER_ORDER_VIOLATION
   A lower credit score has a higher risk tier than a higher credit score.
   Example: score 500 is "super prime" but score 750 is "subprime"

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- Only flag genuine contradictions within the triples themselves.
- Use NO external knowledge — only what the triples say.
- If no inconsistency exists in a group, produce no output for that group.
- If all groups are consistent, return [].
- For MONOTONICITY_VIOLATION: only flag when the higher score has a 
  strictly worse value than the lower score. If values are equal, 
  do NOT flag — equal values are valid and will be verified downstream.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT FORMAT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Return a JSON array. Each object must have exactly these keys:
{
  "type":             "DIRECT_CONFLICT" |
                      "MONOTONICITY_VIOLATION" |
                      "TIER_ORDER_VIOLATION",
  "severity":         "HIGH" | "MEDIUM" | "LOW",
  "explanation":      "One sentence describing the contradiction.",
  "triples_involved": [
    {"subject": "...", "predicate": "...", "object": "..."},
    {"subject": "...", "predicate": "...", "object": "..."}
  ]
}

Severity guide:
  HIGH   — direct contradiction on the same subject (type A)
  MEDIUM — monotonicity or tier ordering violated across subjects (types B and C)
  LOW    — borderline or ambiguous

Output ONLY the JSON array. No markdown, no prose.
"""


# ─────────────────────────────────────────────────────────────────
# Pydantic models
# ─────────────────────────────────────────────────────────────────

from pydantic import BaseModel, model_validator
from typing import Literal, List, Dict

class LogicalInconsistency(BaseModel):
    model_config = {"json_schema_extra": {"required": ["type", "severity", "explanation", "triples_involved"]}}
    
    type: Literal[
        "DIRECT_CONFLICT",
        "MONOTONICITY_VIOLATION",
        "TIER_ORDER_VIOLATION",
    ]
    severity: Literal["HIGH", "MEDIUM", "LOW"]
    explanation: str
    triples_involved: List[Dict[str, str]]


class LogicalInconsistencyResult(BaseModel):
    model_config = {"json_schema_extra": {"required": ["inconsistencies"]}}
    
    inconsistencies: List[LogicalInconsistency]

    @model_validator(mode="before")
    @classmethod
    def wrap_if_list(cls, values):
        if isinstance(values, list):
            return {"inconsistencies": values}
        return values


# ─────────────────────────────────────────────────────────────────
# Group builder — organizes triples, LLM does the comparison
# ─────────────────────────────────────────────────────────────────

def _build_groups(canonicalized: list) -> dict:
    """
    Organizes canonicalized triples into three groups for the LLM to compare.
    No violation detection here — the LLM does all comparison and reasoning.
    """
    from collections import defaultdict

    triples = [item.get("canonical", {}) for item in canonicalized]

    ordered_predicates = MONOTONIC_DECREASING | MONOTONIC_INCREASING
    tier_predicate     = "RISK_TIER_CLASSIFIED_AS_BORROWER_DANGER_LEVEL"

    # ── Group 1: same subject + same predicate, different objects ─
    subj_pred_map = defaultdict(list)
    for t in triples:
        key = (
            str(t.get("subject",   "")).strip().lower(),
            str(t.get("predicate", "")).strip().upper(),
        )
        subj_pred_map[key].append(t)

    same_subject_same_predicate = [
        {
            "subject":   ts[0]["subject"],
            "predicate": ts[0]["predicate"],
            "triples":   ts,
        }
        for (_, __), ts in subj_pred_map.items()
        if len(ts) > 1
    ]

    # ── Group 2: same predicate, different score subjects ─────────
    # Includes ordered predicates and tier predicate
    # Annotated with required direction so LLM knows what to check
    pred_map = defaultdict(list)
    for t in triples:
        pred = str(t.get("predicate", "")).strip().upper()
        if pred in ordered_predicates or pred == tier_predicate:
            pred_map[pred].append(t)

    same_predicate_different_scores = []
    for pred, ts in pred_map.items():
        # Only include if there are multiple different score subjects
        subjects = {str(t.get("subject", "")).strip().lower() for t in ts}
        if len(subjects) < 2:
            continue

        if pred in MONOTONIC_DECREASING:
            direction = "DECREASING"
        elif pred in MONOTONIC_INCREASING:
            direction = "INCREASING"
        else:
            direction = "TIER"

        same_predicate_different_scores.append({
            "predicate":          pred,
            "required_direction": direction,
            "triples":            ts,
        })

    return {
        "same_subject_same_predicate":      same_subject_same_predicate,
        "same_predicate_different_scores":  same_predicate_different_scores,
        "all_triples":                      triples,
    }


# ─────────────────────────────────────────────────────────────────
# STEP 3 — detect_logical_inconsistencies
# ─────────────────────────────────────────────────────────────────

def detect_logical_inconsistencies(state: dict) -> dict:
    canonicalized = state.get("canonicalized_triples", [])
    if not canonicalized:
        state["logical_inconsistencies"] = []
        return state

    groups = _build_groups(canonicalized)

    # Skip LLM if no candidates in any group
    if (not groups["same_subject_same_predicate"] and
        not groups["same_predicate_different_scores"]):
        print("[detect_logical_inconsistencies] no candidates — skipping LLM")
        state["logical_inconsistencies"] = []
        return state

    messages = [
        SystemMessage(content=LOGICAL_INCONSISTENCY_SYSTEM_PROMPT),
        HumanMessage(content=json.dumps(groups, indent=2)),
    ]

    inconsistencies = []
    try:
            raw_response = llm_reasoning_glm.invoke(messages)
            raw_text     = _strip_fences(raw_response.content)
            raw_list     = _safe_parse_list(raw_text, "detect_logical_inconsistencies")
            for item in raw_list:
                inconsistencies.append({
                    "type":             item.get("type",             "DIRECT_CONFLICT"),
                    "severity":         item.get("severity",         "MEDIUM"),
                    "explanation":      item.get("explanation",      ""),
                    "triples_involved": item.get("triples_involved", []),
                })
            print(f"[detect_logical_inconsistencies] GLM fallback: {len(inconsistencies)} inconsistency/ies")
    except Exception as e2:
            print(f"[detect_logical_inconsistencies] fallback also failed: {e2}")
        

    for inc in inconsistencies:
        inc["label"] = "logically_inconsistent"
        print(f"  → [{inc['severity']}] {inc['type']}: {inc['explanation']}")

    state["logical_inconsistencies"] = inconsistencies
    return state

In [47]:
# ─────────────────────────────────────────────────────────────────
# STEP 4 — fetch_neo4j_evidence
# ─────────────────────────────────────────────────────────────────
_global_constraints = None
def fetch_neo4j_evidence(state: dict) -> dict:
    """
    Retrieve ground-truth DB candidates for every canonical triple.
    """
    canonicalized = state.get("canonicalized_triples", [])
    evidence_list = []

    with driver.session() as session:
        for item in canonicalized:

            # LOW-confidence triples skipped — no DB lookup needed
            # LOW-confidence triples: no DB lookup, but still passed to
        # hallucination detection with predicate_known flag so they
        # can be classified as FABRICATED_FACT or UNVERIFIABLE.
            if item.get("match_confidence") == "LOW":
                canonical_pred = item.get("canonical", {}).get("predicate", "")
                global_rel_types = _get_global_relationship_types()
                pred_known = canonical_pred in global_rel_types
                evidence_list.append({
                    "original_triple":  item["original"],
                    "canonical_triple": item["canonical"],
                    "match_confidence": "LOW",
                    "predicate_known":  pred_known,
                    "search_strategy":  None,
                    "db_candidates":    [],
                })
                continue

            t         = item["canonical"]
            subject   = str(t.get("subject",   ""))
            predicate = str(t.get("predicate", ""))
            obj       = str(t.get("object",    ""))

            # ── All 4 lookup strategies in one round-trip ─────────────────────
            cypher = """
            // Q1 — subject + predicate  (strongest)
            OPTIONAL MATCH (n1)-[r1]->(m1)
            WHERE type(r1) = $predicate
            AND NOT 'Constraint' IN labels(n1)
            AND NOT 'Constraint' IN labels(m1)
            AND NOT 'Domain' IN labels(n1)
            AND any(p IN keys(n1) WHERE
                n1[p] IS NOT NULL
                AND NOT (n1[p] = true OR n1[p] = false)
                AND size([x IN [n1[p]] WHERE x = x]) > 0
                AND toLower(toString(n1[p])) = toLower($subject)
            )
            WITH collect({
                subj_props: properties(n1),
                predicate:  type(r1),
                obj_props:  properties(m1),
                strategy:   'subject+predicate'
            }) AS q1

            // Q2 — subject only
            OPTIONAL MATCH (n2)-[r2]->(m2)
            WHERE NOT 'Constraint' IN labels(n2)
            AND NOT 'Constraint' IN labels(m2)
            AND NOT 'Domain' IN labels(n2)
            AND any(p IN keys(n2) WHERE
                n2[p] IS NOT NULL
                AND NOT (n2[p] = true OR n2[p] = false)
                AND toLower(toString(n2[p])) = toLower($subject)
            )
            WITH q1, collect({
                subj_props: properties(n2),
                predicate:  type(r2),
                obj_props:  properties(m2),
                strategy:   'subject_only'
            }) AS q2

            // Q3 — object + predicate
            OPTIONAL MATCH (n3)-[r3]->(m3)
            WHERE type(r3) = $predicate
            AND NOT 'Constraint' IN labels(n3)
            AND NOT 'Constraint' IN labels(m3)
            AND NOT 'Domain' IN labels(n3)
            AND any(p IN keys(m3) WHERE
                m3[p] IS NOT NULL
                AND NOT (m3[p] = true OR m3[p] = false)
                AND toLower(toString(m3[p])) = toLower($obj)
            )
            WITH q1, q2, collect({
                subj_props: properties(n3),
                predicate:  type(r3),
                obj_props:  properties(m3),
                strategy:   'object+predicate'
            }) AS q3

            RETURN q1, q2, q3
            """

            row = session.run(
                cypher,
                subject=subject,
                predicate=predicate,
                obj=obj,
            ).single()

            # Pick the highest-priority non-empty result
            strategy, records = None, []
            if row:
                for key in ("q1", "q2", "q3"):
                    candidates = [
                        c for c in (row[key] or [])
                        if c.get("subj_props") is not None
                    ]
                    if candidates:
                        strategy = candidates[0]["strategy"]
                        records  = candidates
                        break

            # In fetch_neo4j_evidence, add predicate_known flag to evidence entry:
            evidence_list.append({
                    "original_triple":  item["original"],
                    "canonical_triple": item["canonical"],
                    "match_confidence": item["match_confidence"],
                    "predicate_known":  True,   # non-LOW always canonicalized to real DB type
                    "search_strategy":  strategy,
                    "db_candidates":    records,
                })

    state["neo4j_evidence"] = evidence_list
    global _global_constraints
    if _global_constraints is not None:
        return _global_constraints
    with driver.session() as session:
        result = session.run("""
            MATCH (c:Constraint)
            RETURN properties(c) AS constraint
        """)
        _global_constraints = [record["constraint"] for record in result]
    print(f"[fetch_domain_constraints] loaded {len(_global_constraints)} constraints")
    return state

In [48]:
# ─────────────────────────────────────────────────────────────────
# COMBINED AGENT — symbolic_verification + detect_hallucinations
#
# Replaces the two separate agents. Every triple is evaluated in
# two passes inside this single function:
#
#   PASS 1 — Symbolic Constraint Check
#     • Pre-filters constraints by predicate (exact match).
#     • Triples that violate a constraint are labelled:
#         label              = "factually_inconsistent"
#         inconsistency_type = "RULE_VIOLATION"
#       and removed from further checking.
#     • Triples with no matching constraint auto-pass to Pass 2.
#
#   PASS 2 — Hallucination / Factual Verification
#     • Compares each surviving triple against Neo4j evidence.
#     • Verdict → label + inconsistency_type mapping:
#         "CORRECT"              → "factually_consistent"
#         "RULE_VIOLATION"       → "factually_inconsistent", "RULE_VIOLATION"
#         "MULTI_RULE_VIOLATION" → "factually_inconsistent", "MULTI_RULE_VIOLATION"
#         "PREDICATE_MISMATCH"   → "factually_inconsistent", "PREDICATE_MISMATCH"
#         "FABRICATED_FACT"      → "factually_inconsistent", "FABRICATED_FACT"
#         "UNVERIFIABLE"         → "unverifiable"
#   
#
# State keys written:
#   state["symbolic_violations"]   — Pass-1 violating triples
#   state["hallucination_report"]  — Pass-2 verdict dicts
#   state["verification_results"]  — unified per-triple label list
#   state["canonicalized_triples"] — triples surviving Pass 1
# ─────────────────────────────────────────────────────────────────

import json
from typing import List, Dict, Any, Literal, Optional
from pydantic import BaseModel, Field, model_validator


# ─────────────────────────────────────────────────────────────────
# PASS 1 — Symbolic helpers
# ─────────────────────────────────────────────────────────────────
class SymbolicViolation(BaseModel):
    triple: Dict[str, str]
    constraint_id: str = ""
    explanation: str = ""

class PassedTriple(BaseModel):
    triple: Dict[str, str]
    explanation: str = ""

class SymbolicCheckResult(BaseModel):
    violations: List[SymbolicViolation]
    passed: List[PassedTriple]

class TripleClassification(BaseModel):
    subject:       str
    predicate:     str
    object:        str
    passed:        bool  = Field(description="True if triple passes all constraints, False if it violates any")
    constraint_id: str   = Field(default="", description="ID of the violated constraint, empty if passed")
    explanation:   str   = Field(default="", description="One sentence explaining the decision")

    @model_validator(mode="before")
    @classmethod
    def coerce_to_string(cls, values):
        for field in ("subject", "predicate", "object", "constraint_id", "explanation"):
            if field in values and not isinstance(values[field], str):
                values[field] = str(values[field])
        return values


class ClassificationResult(BaseModel):
    classifications: List[TripleClassification]



SYMBOLIC_CHECK_SYSTEM_PROMPT = """
You are a domain constraint checker. You receive a list of items, each containing:
- "triple": {subject, predicate, object}
- "constraints": a list of rules that have the exact same predicate as the triple.

For each item, classify the triple as passed or failed:

1. skip triple if object is literally "", "n/a", or "unknown" or "not in context" "not applicable" or anything that menas
no answer is provided and other smenatic meanings.Make it passed : true
Do NOT skip numeric estimates or extrapolated values — those MUST be evaluated.

2. Evaluate the triple against every constraint in its list.
   - If it violates ANY constraint → passed=false, fill constraint_id and explanation.
   - If it passes ALL constraints → passed=true, fill explanation.

Return a JSON object:
{
  "classifications": [
    {
      "subject":       "<verbatim from triple>",
      "predicate":     "<verbatim from triple>",
      "object":        "<verbatim from triple>",
      "passed":        true | false,
      "constraint_id": "<id of violated constraint, or empty string if passed>",
      "explanation":   "<one sentence>"
    }
  ]
}

Every triple in the input must appear exactly once in classifications.
Output ONLY valid JSON. No markdown, no prose.
"""


# ─────────────────────────────────────────────────────────────────
# PASS 2 — Hallucination detection prompt
# ─────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────
# PASS 2 — Single hallucination-detection prompt (all lanes)
# ─────────────────────────────────────────────────────────────────

HALLUCINATION_SYSTEM_PROMPT = """You are a hallucination-detection engine.
You receive a batch of triples. Each triple has already been pre-routed by the
system based on predicate_known and search_strategy. Your job is to classify
each triple according to its assigned lane. Never use world knowledge.

Each item contains:
  • triple_to_verify — {subject, predicate, object} to judge
  • lane             — one of "A", "B", "C"
  • db_candidates    — ground-truth DB rows (only present for Lane A)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LANE DEFINITIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

LANE A — predicate_known=True, strategy=subject+predicate
  Context : subject+predicate were matched in the DB; db_candidates are provided.
  Task    : Compare the triple's object against db_candidates.
  Verdicts (only these two):
    CORRECT        — object matches semantically a DB candidate value.
    FABRICATED_FACT — subject+predicate exist but object fabricated the stored value.
  violated_component: "OBJECT" for FABRICATED_FACT, null for CORRECT.
  Confidence: 1.0 exact match | 0.8 minor semantic difference.

LANE B — predicate_known=True, strategy=subject_only
  Context : Subject exists in the DB but no records for this subject+predicate
            combination. No db_candidates are provided.
  Task    : Decide whether the object signals absence or asserts a concrete value.
  Verdicts (only these two):
    CORRECT            — ONLY if the object semantically means the information is
                         absent: e.g. "N/A", "unknown", "not provided",
                         "not in context", "cannot provide answer", or any
                         equivalent phrasing signalling no concrete value.
    PREDICATE_MISMATCH — object asserts a real, concrete value but this predicate
                         is not a valid relation for this subject in the DB.
  violated_component: "PREDICATE" for PREDICATE_MISMATCH, null for CORRECT.
  Confidence: 0.9 object clearly signals absence | 0.9 clear concrete assertion.

LANE C — predicate_known=True, strategy=null
  Context : Predicate exists in the DB schema but no DB records were found for
            any search strategy. No db_candidates are provided.
  Task    : Decide whether the object signals absence or asserts a concrete value.
  Verdicts (only these two):
    CORRECT         — ONLY if the object semantically means the information is
                      absent (same criteria as Lane B above).
    FABRICATED_FACT — object asserts a concrete value unsupported by any DB evidence.
  violated_component: "MULTIPLE" for FABRICATED_FACT, null for CORRECT.
  Confidence: 0.9 object clearly signals absence | 0.9 concrete assertion.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MATCHING RULES (Lane A only)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STRICT   — numbers, codes, IDs, percentages → must match exactly.
  SEMANTIC — descriptive labels, names        → match if same meaning.
  NONE     — no match found.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Return a JSON array with one object per triple:
{
  "triple"             : <triple_to_verify verbatim>,
  "verdict"            : <label from the allowed set for the triple's lane>,
  "match_type"         : <"STRICT" | "SEMANTIC" | "NONE">,
  "violated_component" : <"SUBJECT" | "PREDICATE" | "OBJECT" | "MULTIPLE" | null>,
  "confidence"         : <float 0.0-1.0>,
  "evidence"           : <verbatim DB candidate(s) used, or null if no db_candidates>,
  "explanation"        : <one sentence>
}

Rules:
  • You must produce exactly one output object per input item, in the same order.
  • For Lanes B and C: evidence must be null (no DB candidates were provided).
  • For Lane A: evidence must quote actual DB candidates, never paraphrase.
  • Output ONLY a valid JSON array. No markdown, no prose."""


# ─────────────────────────────────────────────────────────────────
# LABEL MAPPING  (verdict → label + inconsistency_type)
# ─────────────────────────────────────────────────────────────────
VERDICT_TO_LABEL = {
    "CORRECT":              ("factually_consistent",   None),
    "RULE_VIOLATION":       ("factually_inconsistent", "RULE_VIOLATION"),
    "PREDICATE_MISMATCH":   ("factually_inconsistent", "PREDICATE_MISMATCH"),
    "FABRICATED_FACT":      ("factually_inconsistent", "FABRICATED_FACT"),
    "UNVERIFIABLE":         ("unverifiable",           None),
}


# ─────────────────────────────────────────────────────────────────
# COMBINED AGENT
# ─────────────────────────────────────────────────────────────────

def combined_verification_agent(state: dict) -> dict:
    """
    Pass 1 — Symbolic constraint check (pre-filters by predicate match).
    Pass 2 — Hallucination / factual detection against Neo4j evidence.

    State keys written:
        symbolic_violations   — Pass-1 violating triples
                                (label="factually_inconsistent",
                                 inconsistency_type="RULE_VIOLATION")
        hallucination_report  — Pass-2 verdict dicts with label+inconsistency_type
        verification_results  — unified per-triple list
        canonicalized_triples — triples surviving Pass 1
    """

    canonicalized = state.get("canonicalized_triples", [])
    if not canonicalized:
        state["symbolic_violations"]  = []
        state["hallucination_report"] = []
        state["verification_results"] = []
        return state

    # ══════════════════════════════════════════════════════════════
    # PASS 1 — SYMBOLIC CONSTRAINT CHECK
    # ══════════════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("COMBINED AGENT — PASS 1: Symbolic Constraint Check")
    print("=" * 60)

    constraints = _global_constraints

    triples_with_constraints = []
    auto_passed_triples      = []

    for item in canonicalized:
        triple = item.get("canonical", {})
        pred   = triple.get("predicate", "").strip().lower()
        if not pred:
            auto_passed_triples.append(triple)
            continue
        matching = [c for c in constraints
                    if c.get("predicate", "").strip().lower() == pred]
        if not matching:
            auto_passed_triples.append(triple)
        else:
            triples_with_constraints.append({"triple": triple, "constraints": matching})

    if triples_with_constraints:
        print("\nTriples checked by LLM (Pass 1):\n")
        for idx, entry in enumerate(triples_with_constraints, 1):
            t = entry["triple"]
            print(f"  {idx}. ({t.get('subject','')}, {t.get('predicate','')}, "
                  f"{t.get('object','')}) — {len(entry['constraints'])} constraint(s)")
    if auto_passed_triples:
        print("\nTriples auto-passed (no matching constraints):\n")
        for idx, t in enumerate(auto_passed_triples, 1):
            print(f"  {idx}. ({t.get('subject','')}, {t.get('predicate','')}, {t.get('object','')})")
    print("=" * 60)

    all_symbolic_violations = []
    all_symbolic_passed     = []

    if triples_with_constraints:
        BATCH_SIZE      = 10
        system_msg      = SystemMessage(content=SYMBOLIC_CHECK_SYSTEM_PROMPT)
        symbolic_chain  = llm_reasoning_gpt.with_structured_output(
            ClassificationResult, method="json_mode"
        )
        total_batches = (len(triples_with_constraints) + BATCH_SIZE - 1) // BATCH_SIZE

        for batch_num, batch_start in enumerate(
            range(0, len(triples_with_constraints), BATCH_SIZE), 1
        ):
            batch    = triples_with_constraints[batch_start: batch_start + BATCH_SIZE]
            messages = [system_msg,
                        HumanMessage(content=json.dumps({"items": batch}, indent=2))]
            print(f"[Pass 1] Batch {batch_num}/{total_batches} — {len(batch)} triple(s)")
            try:
                result          = symbolic_chain.invoke(messages)
                classifications = result.classifications
            except Exception as e:
                print(f"[Pass 1] Batch {batch_num} failed: {e} — fallback")
                try:
                    response        = llm_reasoning_gemini.invoke(messages)
                    raw             = _strip_fences(response.content)
                    parsed          = json.loads(raw)
                    items           = parsed.get("classifications", parsed if isinstance(parsed, list) else [])
                    classifications = [TripleClassification(**item) for item in items]
                except Exception as e2:
                    print(f"[Pass 1] Batch {batch_num} fallback failed: {e2} — skipping")
                    classifications = []

            # ── Code does the sorting, not the LLM ──
            batch_violations = []
            batch_passed     = []
            for c in classifications:
                triple_dict = {
                    "subject":   c.subject,
                    "predicate": c.predicate,
                    "object":    c.object,
                }
                if not c.passed:
                    batch_violations.append({
                        "triple":        triple_dict,
                        "constraint_id": c.constraint_id,
                        "explanation":   c.explanation,
                    })
                else:
                    batch_passed.append({
                        "triple":      triple_dict,
                        "explanation": c.explanation,
                    })

            print(f"  -> {len(batch_violations)} violation(s), {len(batch_passed)} passed")
            all_symbolic_violations.extend(batch_violations)
            all_symbolic_passed.extend(batch_passed)

    # Remove any passed triple that also appears in violations
    violation_keys = {
        (v.get("triple", {}).get("subject",   ""),
         v.get("triple", {}).get("predicate", ""),
         v.get("triple", {}).get("object",    ""))
        for v in all_symbolic_violations
    }
    all_symbolic_passed = [
        p for p in all_symbolic_passed
        if (p.get("triple", {}).get("subject",   ""),
            p.get("triple", {}).get("predicate", ""),
            p.get("triple", {}).get("object",    "")) not in violation_keys
    ]
    for t in auto_passed_triples:
        all_symbolic_passed.append({
            "triple":      t,
            "explanation": "No matching constraint found (automatically passed).",
        })

    # Label Pass-1 violations as factually_inconsistent
    for v in all_symbolic_violations:
        v["label"]              = "factually_inconsistent"
        v["inconsistency_type"] = "RULE_VIOLATION"

    print(f"\n[Pass 1 Summary] {len(all_symbolic_violations)} violation(s), "
          f"{len(all_symbolic_passed)} passed")

    # Survivors for Pass 2
    passed_keys = {
        (p.get("triple", {}).get("subject",   ""),
         p.get("triple", {}).get("predicate", ""),
         p.get("triple", {}).get("object",    ""))
        for p in all_symbolic_passed
    }
    state["canonicalized_triples"] = [
        item for item in canonicalized
        if (item.get("canonical", {}).get("subject",   ""),
            item.get("canonical", {}).get("predicate", ""),
            item.get("canonical", {}).get("object",    "")) in passed_keys
    ]
    state["symbolic_violations"] = all_symbolic_violations

    # ══════════════════════════════════════════════════════════════
    # PASS 2 — HALLUCINATION / FACTUAL DETECTION
    # ══════════════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("COMBINED AGENT — PASS 2: Hallucination / Factual Detection")
    print("=" * 60)

    def flatten_props(props) -> str:
        if not isinstance(props, dict):
            return str(props) if props is not None else ""
        if not props:
            return ""
        if len(props) == 1:
            return str(next(iter(props.values())))
        for lo, hi in [("min", "max"), ("from", "to"), ("start", "end"),
                       ("lower", "upper"), ("low", "high")]:
            if lo in props and hi in props and len(props) == 2:
                return f"{props[lo]}-{props[hi]}"
        return ", ".join(f"{k}: {v}" for k, v in props.items())

    evidence_list = state.get("neo4j_evidence", [])

    # ── Route every surviving triple into a verdict or an LLM item ─
    auto_verdicts = []   # deterministic — no LLM needed
    llm_items     = []   # will be sent in one batched LLM call

    for item in evidence_list:
        ct         = item["canonical_triple"]
        triple_key = (ct.get("subject", ""), ct.get("predicate", ""), ct.get("object", ""))
        if triple_key not in passed_keys:
            continue  # already flagged in Pass 1

        pred_known = item.get("predicate_known", False)
        strategy   = item.get("search_strategy")  # "subject+predicate" | "subject_only" | "object+predicate" | None

        # ── Rule 1: predicate_known=False → UNVERIFIABLE (no LLM) ──
        if not pred_known:
            auto_verdicts.append({
                "triple"             : ct,
                "verdict"            : "UNVERIFIABLE",
                "match_type"         : "NONE",
                "violated_component" : None,
                "confidence"         : 0.0,
                "evidence"           : None,
                "explanation"        : "Predicate is not present in the DB schema; triple cannot be verified.",
            })
            print(f"  [Pass 2] AUTO → UNVERIFIABLE (predicate unknown): "
                  f"({ct.get('subject')}, {ct.get('predicate')}, {ct.get('object')})")
            continue

        # predicate_known=True from here on ─────────────────────────

        # ── Rule 2: strategy="object+predicate" → FABRICATED_FACT (no LLM) ──
        if strategy == "object+predicate":
            auto_verdicts.append({
                "triple"             : ct,
                "verdict"            : "FABRICATED_FACT",
                "match_type"         : "NONE",
                "violated_component" : "SUBJECT",
                "confidence"         : 0.9,
                "evidence"           : None,
                "explanation"        : (
                    "Object+predicate exist in DB but no matching subject found; "
                    "subject is fabricated."
                ),
            })
            print(f"  [Pass 2] AUTO → FABRICATED_FACT (object+predicate, subject absent): "
                  f"({ct.get('subject')}, {ct.get('predicate')}, {ct.get('object')})")
            continue

        # ── LLM lanes: build item with lane tag ────────────────────
        if strategy == "subject+predicate":
            flattened = [
                {
                    "subject"  : flatten_props(c.get("subj_props", {})),
                    "predicate": str(c.get("predicate", "")),
                    "object"   : flatten_props(c.get("obj_props", {})),
                }
                for c in item.get("db_candidates", [])
            ]
            llm_items.append({
                "triple_to_verify": ct,
                "lane"            : "A",
                "db_candidates"   : flattened,
            })

        elif strategy == "subject_only":
            llm_items.append({
                "triple_to_verify": ct,
                "lane"            : "B",
                "db_candidates"   : [],
            })

        else:  # null or any unexpected value
            llm_items.append({
                "triple_to_verify": ct,
                "lane"            : "C",
                "db_candidates"   : [],
            })

    # ── Single batched LLM call ─────────────────────────────────────
    llm_verdicts = []
    if llm_items:
        BATCH_SIZE    = 10
        total_batches = (len(llm_items) + BATCH_SIZE - 1) // BATCH_SIZE
        for i in range(0, len(llm_items), BATCH_SIZE):
            batch     = llm_items[i: i + BATCH_SIZE]
            batch_num = i // BATCH_SIZE + 1
            messages  = [
                SystemMessage(content=HALLUCINATION_SYSTEM_PROMPT),
                HumanMessage(content=json.dumps(batch, indent=2)),
            ]
            response = llm_reasoning_gpt.invoke(messages)
            print(f"  [Pass 2] Batch {batch_num}/{total_batches} preview: "
                  f"{repr(response.content[:200])}")
            batch_verdicts = _safe_parse_list(
                _strip_fences(response.content),
                f"combined_agent.pass2.batch{batch_num}",
            )
            llm_verdicts.extend(batch_verdicts)

    all_verdicts = auto_verdicts + llm_verdicts

    print(f"\n[Pass 2] Deterministic: {len(auto_verdicts)} | "
          f"LLM (A+B+C): {len(llm_verdicts)} | "
          f"Total: {len(all_verdicts)}")

    # Attach label + inconsistency_type to each verdict
    for v in all_verdicts:
        verdict      = v.get("verdict", "UNVERIFIABLE")
        label, itype = VERDICT_TO_LABEL.get(verdict, ("unverifiable", None))
        v["label"]   = label
        if itype:
            v["inconsistency_type"] = itype

    state["hallucination_report"] = all_verdicts

    # ══════════════════════════════════════════════════════════════
    # UNIFIED VERIFICATION RESULTS
    # ══════════════════════════════════════════════════════════════
    unified = []

    for sv in all_symbolic_violations:
        t = sv.get("triple", {})
        unified.append({
            "subject":            t.get("subject",   ""),
            "predicate":          t.get("predicate", ""),
            "object":             t.get("object",    ""),
            "label":              sv.get("label",              "factually_inconsistent"),
            "inconsistency_type": sv.get("inconsistency_type", "RULE_VIOLATION"),
            "source":             "symbolic_pass1",
            "explanation":        sv.get("explanation", ""),
            "constraint_id":      sv.get("constraint_id", ""),
        })

    for v in all_verdicts:
        raw  = v.get("triple") or v.get("triple_to_verify", {})
        subj = raw.get("subject",   "") if isinstance(raw, dict) else ""
        pred = raw.get("predicate", "") if isinstance(raw, dict) else ""
        obj  = raw.get("object",    "") if isinstance(raw, dict) else ""
        entry = {
            "subject":    subj,
            "predicate":  pred,
            "object":     obj,
            "label":      v.get("label", "unverifiable"),
            "source":     "hallucination_pass2",
            "verdict":    v.get("verdict", "UNVERIFIABLE"),
            "confidence": v.get("confidence", 0.0),
            "explanation":v.get("explanation", ""),
            "evidence":   v.get("evidence", ""),
        }
        if "inconsistency_type" in v:
            entry["inconsistency_type"] = v["inconsistency_type"]
        unified.append(entry)

    state["verification_results"] = unified
    # Print summary
    print("\n" + "=" * 60)
    print("COMBINED AGENT — FINAL SUMMARY")
    print("=" * 60)
    label_counts: dict = {}
    for u in unified:
        k = u["label"] + (": " + u.get("inconsistency_type", "")
                          if u.get("inconsistency_type") else "")
        label_counts[k] = label_counts.get(k, 0) + 1
    for k, cnt in sorted(label_counts.items()):
        print(f"  {k:<55} {cnt}")
    print("=" * 60)

    return state


In [49]:
def generate_report_pdf(state: dict) -> dict:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib import colors
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch, mm
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
        HRFlowable, PageBreak, KeepTogether, FrameBreak,
    )
    from reportlab.platypus.flowables import Flowable
    from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
    from reportlab.pdfgen import canvas as rl_canvas
    import os
    from datetime import datetime

    # ── Parse hallucination_report ────────────────────────────────
    report_raw = state.get("hallucination_report", [])
    if isinstance(report_raw, list):
        verdicts = report_raw
    elif isinstance(report_raw, str) and report_raw.strip():
        # Assuming _safe_parse_list and _strip_fences are defined in your scope
        verdicts = _safe_parse_list(_strip_fences(report_raw), "generate_report_pdf")
    else:
        verdicts = []

    if verdicts and isinstance(verdicts[0], list):
        verdicts = [item for sublist in verdicts for item in sublist]
    verdicts = [v for v in verdicts if isinstance(v, dict)]

    # ── Palette ───────────────────────────────────────────────────
    WHITE     = colors.HexColor("#FFFFFF")
    OFF_WHITE = colors.HexColor("#F8F9FB")
    NAVY_900  = colors.HexColor("#0A1628")
    NAVY_700  = colors.HexColor("#142847")
    NAVY_500  = colors.HexColor("#1E3A5F")
    NAVY_100  = colors.HexColor("#E8EDF5")
    INK_900   = colors.HexColor("#0D1117")
    INK_600   = colors.HexColor("#374151")
    INK_400   = colors.HexColor("#6B7280")
    INK_200   = colors.HexColor("#9CA3AF")
    RULE      = colors.HexColor("#E2E8F0")
    RULE_MID  = colors.HexColor("#CBD5E1")

    VRD = {
        "CORRECT":              ("#065F46", "#ECFDF5", "#6EE7B7"),
        "RULE_VIOLATION":       ("#7F1D1D", "#FEF2F2", "#FCA5A5"),
        "UNVERIFIABLE":         ("#1E40AF", "#EFF6FF", "#93C5FD"),
        "PREDICATE_MISMATCH":   ("#4C1D95", "#F5F3FF", "#C4B5FD"),
        "FABRICATED_FACT":      ("#374151", "#F9FAFB", "#D1D5DB"),
    }
    VERDICT_LABELS = {
        "CORRECT":              "Correct",
        "RULE_VIOLATION":       "Rule Violation",
        "UNVERIFIABLE":         "Unverifiable",
        "PREDICATE_MISMATCH":   "Predicate Mismatch",
        "FABRICATED_FACT":      "Fabricated Fact",
    }
    LEGEND_DESC = {
        "CORRECT":              "All three triple components verified against the knowledge graph.",
        "RULE_VIOLATION":       "One or more components contradict database evidence rule.",
        "UNVERIFIABLE":         "No usable DB evidence found for any component of this triple.",
        "PREDICATE_MISMATCH":   "Subject entity exists, but the stated predicate is not a valid relation.",
        "FABRICATED_FACT":      "Triple asserts a relationship that is in db but the subject, object or subject+object does not supported by db.",
    }
    ALL_VERDICTS = list(VRD.keys())

    SEV = {
        "HIGH":   ("#7F1D1D", "#FEF2F2", "#FCA5A5"),
        "MEDIUM": ("#78350F", "#FFFBEB", "#FCD34D"),
        "LOW":    ("#1E40AF", "#EFF6FF", "#93C5FD"),
    }
    CONF = {
        "HIGH":   ("#065F46", "#ECFDF5", "#6EE7B7"),
        "MEDIUM": ("#78350F", "#FFFBEB", "#FCD34D"),
        "LOW":    ("#7F1D1D", "#FEF2F2", "#FCA5A5"),
    }
    LBL = {
        "factually_consistent":   ("#065F46", "#ECFDF5", "#6EE7B7"),
        "factually_inconsistent": ("#7F1D1D", "#FEF2F2", "#FCA5A5"),
        "logically_inconsistent": ("#4C1D95", "#F5F3FF", "#C4B5FD"),
        "unverifiable":           ("#1E40AF", "#EFF6FF", "#93C5FD"),
    }

    def hex_color(h): return colors.HexColor(h)

    # ── Output path ───────────────────────────────────────────────
    output_path = os.path.join(
        os.path.expanduser("~"),
        "Desktop/Bachelor/Methadology",
        "hallucination_report.pdf",
    )
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # ── Page geometry ─────────────────────────────────────────────
    PAGE_W, PAGE_H = A4
    L_MAR = 0.8 * inch
    R_MAR = 0.8 * inch
    T_MAR = 0.80 * inch
    B_MAR = 0.65 * inch
    W = PAGE_W - L_MAR - R_MAR

    run_time = datetime.now().strftime("%Y-%m-%d  %H:%M UTC")

    def on_page(canv, doc):
        canv.saveState()
        canv.setFillColor(NAVY_900)
        canv.rect(0, PAGE_H - 6, PAGE_W, 6, fill=1, stroke=0)
        canv.setFillColor(hex_color("#2563EB"))
        canv.rect(0, PAGE_H - 9, PAGE_W, 3, fill=1, stroke=0)
        if doc.page > 1:
            canv.setFont("Helvetica", 6)
            canv.setFillColor(INK_400)
            canv.drawString(L_MAR, PAGE_H - 21, "HALLUCINATION DETECTION REPORT")
            canv.drawRightString(PAGE_W - R_MAR, PAGE_H - 21, run_time)
            canv.setStrokeColor(RULE)
            canv.setLineWidth(0.5)
            canv.line(L_MAR, PAGE_H - 25, PAGE_W - R_MAR, PAGE_H - 25)
        canv.setStrokeColor(RULE)
        canv.setLineWidth(0.5)
        canv.line(L_MAR, B_MAR - 6, PAGE_W - R_MAR, B_MAR - 6)
        canv.setFont("Helvetica", 6)
        canv.setFillColor(INK_400)
        canv.drawString(L_MAR, B_MAR - 15, "KGVS: Neuro-Symbolic AI Verification")
        canv.drawRightString(PAGE_W - R_MAR, B_MAR - 15, f"Page {doc.page}")
        canv.restoreState()

    # ── Styles ─────────────────────────────────
    styles = getSampleStyleSheet()
    def S(name, **kw): 
        # Pop the parent out of kwargs if it exists, otherwise default to 'Normal'
        parent_style = kw.pop("parent", styles["Normal"])
        return ParagraphStyle(name, parent=parent_style, **kw)

    # Cover Styles
    cov_uni   = S("CUni", fontSize=14, fontName="Helvetica-Bold", textColor=NAVY_900, alignment=TA_CENTER, spaceAfter=4)
    cov_dept  = S("CDept", fontSize=11, fontName="Helvetica", textColor=INK_600, alignment=TA_CENTER, spaceAfter=8)
    cov_title = S("CTitle", fontSize=26, fontName="Helvetica-Bold", textColor=NAVY_900, alignment=TA_CENTER, leading=32, spaceAfter=12)
    cov_sub   = S("CSub", fontSize=12, fontName="Helvetica-Oblique", textColor=NAVY_500, alignment=TA_CENTER, spaceAfter=40)
    cov_auth  = S("CAuth", fontSize=12, fontName="Helvetica-Bold", textColor=NAVY_900, alignment=TA_CENTER, spaceAfter=4)
    cov_meta  = S("CMeta", fontSize=10, fontName="Helvetica", textColor=INK_400, alignment=TA_CENTER)

    # Body Styles
    note_style  = S("NT", fontSize=8, textColor=INK_400, spaceAfter=8, leading=12)
    p_body      = S("PB", fontSize=9, textColor=INK_900, spaceAfter=10, leading=14, alignment=TA_LEFT)
    cell_style  = S("C",  fontSize=6.5, leading=9.5,  textColor=INK_600, wordWrap="CJK")
    cell_muted  = S("Cm", fontSize=6,   leading=9,    textColor=INK_400, wordWrap="CJK")
    cell_hint   = S("Ch", fontSize=5.5, leading=8,    textColor=INK_200, wordWrap="CJK", fontName="Helvetica-Oblique")
    cell_mono   = S("Mo", fontName="Courier", fontSize=6, textColor=INK_400, leading=9)
    hdr_style   = S("H",  fontSize=6.5, fontName="Helvetica-Bold", textColor=WHITE, alignment=TA_LEFT, leading=9)

    def P(text, style=None, **kw): return Paragraph(str(text), style or cell_style)

    # ── Chip & UI Helpers ───────────────────────────────────────────────
    def chip(label, fg_hex, bg_hex, bd_hex, size=6, bold=True):
        font = "Helvetica-Bold" if bold else "Helvetica"
        para = Paragraph(
            f'<font color="{fg_hex}"><b>{label}</b></font>' if bold else f'<font color="{fg_hex}">{label}</font>',
            S(f"chip_{str(label)[:6]}", fontSize=size, fontName=font, textColor=hex_color(fg_hex), alignment=TA_CENTER, leading=size + 2)
        )
        t = Table([[para]], colWidths=[None])
        t.setStyle(TableStyle([
            ("BACKGROUND",    (0, 0), (-1, -1), hex_color(bg_hex)),
            ("BOX",           (0, 0), (-1, -1), 0.5, hex_color(bd_hex)),
            ("TOPPADDING",    (0, 0), (-1, -1), 2),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
            ("LEFTPADDING",   (0, 0), (-1, -1), 5),
            ("RIGHTPADDING",  (0, 0), (-1, -1), 5),
            ("ROUNDEDCORNERS", [3]),
        ]))
        return t

    def verdict_chip(vrd):
        fg, bg, bd = VRD.get(vrd, ("#374151", "#F9FAFB", "#D1D5DB"))
        return chip(VERDICT_LABELS.get(vrd, vrd), fg, bg, bd)

    def sev_chip(sev):
        fg, bg, bd = SEV.get(sev, ("#374151", "#F9FAFB", "#D1D5DB"))
        return chip(sev, fg, bg, bd)

    def conf_chip(conf):
        fg, bg, bd = CONF.get(conf, ("#374151", "#F9FAFB", "#D1D5DB"))
        return chip(conf, fg, bg, bd)

    def label_chip(lbl, itype=""):
        fg, bg, bd = LBL.get(lbl, ("#374151", "#F9FAFB", "#D1D5DB"))
        text = lbl + ("\n" + itype if itype else "")
        return chip(text, fg, bg, bd, size=5.5)

    def conf_bar(score):
        try:    score = float(score)
        except: score = 0.0
        score  = max(0.0, min(1.0, score))
        filled = round(score * 10)
        return "●" * filled + "○" * (10 - filled) + f"  {score:.2f}"

    def section_heading(text):
        label = Paragraph(text.upper(), S(f"sh_{str(text)[:6]}", fontSize=8, fontName="Helvetica-Bold", textColor=NAVY_500, leading=10, letterSpacing=0.8))
        t = Table([[Spacer(3, 0), label]], colWidths=[3, W - 3])
        t.setStyle(TableStyle([
            ("BACKGROUND",    (0, 0), (0, -1), hex_color("#2563EB")),
            ("TOPPADDING",    (0, 0), (-1, -1), 4),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
            ("LEFTPADDING",   (0, 0), (0, -1),  0),
            ("RIGHTPADDING",  (0, 0), (0, -1),  0),
            ("LEFTPADDING",   (1, 0), (1, -1),  8),
            ("RIGHTPADDING",  (1, 0), (1, -1),  4),
            ("BACKGROUND",    (1, 0), (1, -1), NAVY_100),
            ("VALIGN",        (0, 0), (-1, -1), "MIDDLE"),
        ]))
        return t
    
    def trunc(text, max_chars=300):
        """Truncate long strings so they never blow out a table row."""
        text = str(text or "")
        return text[:max_chars] + "…" if len(text) > max_chars else text

    def std_table_style():
        return TableStyle([
            ("BACKGROUND",    (0, 0), (-1, 0),  NAVY_500),
            ("TOPPADDING",    (0, 0), (-1, -1), 5),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
            ("LEFTPADDING",   (0, 0), (-1, -1), 6),
            ("RIGHTPADDING",  (0, 0), (-1, -1), 6),
            ("VALIGN",        (0, 0), (-1, -1), "TOP"),
            ("LINEBELOW",     (0, 0), (-1, -1), 0.3, RULE),
            ("LINEBEFORE",    (0, 0), (0, -1),  0.3, RULE),
            ("WORDWRAP", (0, 0), (-1, -1), True),
            ("LINEAFTER",     (-1, 0),(-1, -1), 0.3, RULE),
            ("ROWBACKGROUNDS",(0, 1), (-1, -1), [WHITE, OFF_WHITE]),
        ])

    def to_str(val, max_len=100):
        if val is None: return ""
        if isinstance(val, str): return val
        if isinstance(val, (int, float, bool)): return str(val)
        if isinstance(val, dict):
            for key in ["name", "value", "text", "condition", "description", "label"]:
                if key in val: return to_str(val[key], max_len)
            items = [f"{k}: {to_str(v, max_len // 2)}" for k, v in val.items() if not k.startswith("_")]
            result = ", ".join(items)
            return result[:max_len] + "…" if len(result) > max_len else result
        if isinstance(val, list):
            return ";  ".join(to_str(item, max_len) for item in val)
        return str(val)

    def parse_triple(v):
        raw = v.get("triple") or v.get("triple_to_verify")
        if isinstance(raw, dict):
            return (to_str(raw.get("subject", "")), to_str(raw.get("predicate", "")), to_str(raw.get("object", "")))
        if isinstance(raw, list) and len(raw) == 3:
            return to_str(raw[0]), to_str(raw[1]), to_str(raw[2])
        if isinstance(raw, str) and raw.strip():
            cleaned = raw.strip().lstrip("(").rstrip(")")
            for sep in ["|", " -> ", " → ", "\t"]:
                parts = [p.strip() for p in cleaned.split(sep)]
                if len(parts) == 3: return parts[0], parts[1], parts[2]
            return cleaned, "", ""
        if v.get("subject") or v.get("predicate") or v.get("object"):
            return (to_str(v.get("subject", "")), to_str(v.get("predicate", "")), to_str(v.get("object", "")))
        return "", "", ""

    # ── Sanitize verdicts ─────────────────────────────────────────
    clean_verdicts = []
    for v in verdicts:
        subj, pred, obj = parse_triple(v)
        clean_verdicts.append({
            "subject":            subj,
            "predicate":          pred,
            "object":             obj,
            "verdict":            to_str(v.get("verdict",            "UNVERIFIABLE")),
            "match_type":         to_str(v.get("match_type",         "NONE")),
            "violated_component": to_str(v.get("violated_component", "")),
            "confidence":         v.get("confidence", 0.0),
            "evidence":           to_str(v.get("evidence",           "")),
            "explanation":        to_str(v.get("explanation",        "")),
            "label":              to_str(v.get("label",              "")),
            "inconsistency_type": to_str(v.get("inconsistency_type", "")),
        })
    verdicts = clean_verdicts

    extracted_triples       = state.get("extracted_triples",          [])
    canonicalized_triples   = state.get("canonicalized_triples_Before", [])
    logical_inconsistencies = state.get("logical_inconsistencies",    [])
    symbolic_violations     = state.get("symbolic_violations",        [])

    # ── Document ──────────────────────────────────────────────────
    doc = SimpleDocTemplate(
        output_path, pagesize=A4,
        leftMargin=L_MAR, rightMargin=R_MAR,
        topMargin=T_MAR,  bottomMargin=B_MAR,
        title="Hallucination Detection Report",
        author="Mohammed Mahmoud Elnaggar",
    )
    story = []

    # ══════════════════════════════════════════════════════════════
    # PAGE 1 — ACADEMIC COVER PAGE
    # ══════════════════════════════════════════════════════════════
    story.append(Spacer(1, 40))
    story.append(P("German University in Cairo (GUC)", cov_uni))
    story.append(P("Media Engineering and Technology (MET)", cov_dept))
    story.append(Spacer(1, 80))
    
    story.append(P("Automated Hallucination Detection Report", cov_title))
    story.append(P("Neuro-Symbolic Verification via Knowledge Graphs", cov_sub))
    
    story.append(Spacer(1, 100))
    story.append(P("Bachelor Thesis Project Output", cov_meta))
    story.append(Spacer(1, 10))
    story.append(P("Mohammed Mahmoud Elnaggar", cov_auth))
    story.append(Spacer(1, 20))
    story.append(P(f"Generated on: {run_time}", cov_meta))
    
    story.append(PageBreak())

    # ══════════════════════════════════════════════════════════════
    # PAGE 2 — EXECUTIVE SUMMARY & PIPELINE OVERVIEW
    # ══════════════════════════════════════════════════════════════
    story.append(section_heading("Report Overview & Methodology"))
    story.append(Spacer(1, 10))
    
    story.append(P("This document outlines the results of the KGVS verification pipeline, a neuro-symbolic approach designed to enforce domain rules and verify factual consistency in Black-Box Large Language Models (LLMs).", p_body))
    
    story.append(P("<b>Pipeline Stages:</b>", S("PBold", parent=p_body, fontName="Helvetica-Bold")))
    story.append(P("<b>1. Triple Extraction & Canonicalization:</b> Unstructured LLM outputs are decomposed into relational triples (Subject, Predicate, Object) and mapped to the formal schema of the underlying database.", p_body))
    story.append(P("<b>2. Symbolic Constraint Verification:</b> Triples are subjected to rigorous logical constraint checks to identify domain-rule violations prior to graph traversal.", p_body))
    story.append(P("<b>3. Graph-based Factual Verification:</b> Canonicalized triples are queried against the Neo4j Knowledge Graph to categorize them as strictly correct, unverifiable, or explicitly fabricated.", p_body))
    
    story.append(Spacer(1, 20))
    
    # KPI Strip
    counts = {}
    for v in verdicts:
        k = v.get("verdict", "UNVERIFIABLE")
        counts[k] = counts.get(k, 0) + 1

    kpi_items = [
        ("Triples Extracted",    len(extracted_triples),       "#1E3A5F", "#EFF6FF"),
        ("Final Verdicts",       len(verdicts),                "#065F46", "#ECFDF5"),
        ("Logic Inconsistencies",len(logical_inconsistencies), "#78350F", "#FFFBEB"),
        ("Rule Violations",      len(symbolic_violations),     "#7F1D1D", "#FEF2F2"),
    ]
    
    kpi_cells = []
    for label_kpi, val, fg_h, bg_h in kpi_items:
        cell_inner = Table([[Paragraph(
            f'<font size="17" color="{fg_h}"><b>{val}</b></font><br/>'
            f'<font size="7" color="{INK_400.hexval()}">{label_kpi}</font>',
            S(f"kpi_{label_kpi[:4]}", alignment=TA_CENTER, leading=22)
        )]])
        cell_inner.setStyle(TableStyle([
            ("BACKGROUND",    (0, 0), (-1, -1), hex_color(bg_h)),
            ("ALIGN",         (0, 0), (-1, -1), "CENTER"),
            ("VALIGN",        (0, 0), (-1, -1), "MIDDLE"),
            ("TOPPADDING",    (0, 0), (-1, -1), 12),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 12),
            ("BOX",           (0, 0), (-1, -1), 0.5, RULE_MID),
            ("ROUNDEDCORNERS",[5]),
        ]))
        kpi_cells.append(cell_inner)

    kpi_table = Table([kpi_cells], colWidths=[W / 4] * 4)
    kpi_table.setStyle(TableStyle([
        ("INNERGRID",     (0, 0), (-1, -1), 0, WHITE),
        ("LEFTPADDING",   (0, 0), (-1, -1), 3),
        ("RIGHTPADDING",  (0, 0), (-1, -1), 3),
        ("TOPPADDING",    (0, 0), (-1, -1), 0),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 0),
    ]))
    story.append(kpi_table)
    story.append(Spacer(1, 22))

    # Verdict breakdown
    story.append(section_heading("Verdict Distribution"))
    story.append(Spacer(1, 8))

    active_vrds = [v for v in ALL_VERDICTS if counts.get(v, 0) > 0] or ALL_VERDICTS

    def vrd_card(vrd):
        fg_h, bg_h, bd_h = VRD.get(vrd, ("#374151", "#F9FAFB", "#D1D5DB"))
        lbl = VERDICT_LABELS.get(vrd, vrd)
        cnt = counts.get(vrd, 0)
        pct = f"{cnt / max(len(verdicts), 1) * 100:.0f}%" if verdicts else "—"
        inner = Table([[Paragraph(
            f'<font size="20" color="{fg_h}"><b>{cnt}</b></font><br/>'
            f'<font size="6" color="{INK_400.hexval()}">{pct} of verdicts</font><br/>'
            f'<font size="6.5" color="{fg_h}"><b>{lbl}</b></font>',
            S(f"vc_{vrd[:4]}", alignment=TA_CENTER, leading=24)
        )]])
        inner.setStyle(TableStyle([
            ("BACKGROUND",    (0, 0), (-1, -1), hex_color(bg_h)),
            ("ALIGN",         (0, 0), (-1, -1), "CENTER"),
            ("VALIGN",        (0, 0), (-1, -1), "MIDDLE"),
            ("TOPPADDING",    (0, 0), (-1, -1), 14),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 14),
            ("BOX",           (0, 0), (-1, -1), 0.5, hex_color(bd_h)),
            ("ROUNDEDCORNERS",[5]),
        ]))
        return inner

    cols_per_row = 3
    rows = []
    for i in range(0, len(active_vrds), cols_per_row):
        row_cells = [vrd_card(v) for v in active_vrds[i:i + cols_per_row]]
        while len(row_cells) < cols_per_row:
            row_cells.append(Spacer(1, 1))
        rows.append(row_cells)

    vrd_grid = Table(rows, colWidths=[W / cols_per_row] * cols_per_row)
    vrd_grid.setStyle(TableStyle([
        ("INNERGRID",     (0, 0), (-1, -1), 0, WHITE),
        ("LEFTPADDING",   (0, 0), (-1, -1), 4),
        ("RIGHTPADDING",  (0, 0), (-1, -1), 4),
        ("TOPPADDING",    (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
    ]))
    story.append(vrd_grid)
    story.append(PageBreak())

    # ══════════════════════════════════════════════════════════════
    # DATA SECTIONS (The rest remains exactly the same logic, 
    # ensuring your data formatting works perfectly)
    # ══════════════════════════════════════════════════════════════
    
    # ── EXTRACTED TRIPLES ──
    story.append(section_heading("Extracted Triples"))
    story.append(Spacer(1, 5))
    story.append(P("All verifiable triples extracted from the LLM response prior to canonicalization or knowledge-graph checking.", note_style))
    story.append(Spacer(1, 6))

    if extracted_triples:
        ext_data = [[P(h, hdr_style) for h in ["#", "Subject", "Predicate", "Object", "Type", "Predicate Definition"]]]
        for i, t in enumerate(extracted_triples, 1):
            ext_data.append([
                P(str(i), S("idx", fontSize=5.5, textColor=INK_200, alignment=TA_CENTER, leading=9)),
                P(to_str(t.get("subject",              "")), cell_style),
                P(to_str(t.get("predicate",            "")), cell_muted),
                P(to_str(t.get("object",               "")), cell_style),
                P(to_str(t.get("verification_type",    "")), cell_hint),
                P(to_str(t.get("predicate_definition", "")), cell_hint),
            ])
        ext_t = Table(ext_data, colWidths=[W*0.04, W*0.13, W*0.17, W*0.13, W*0.12, W*0.41], repeatRows=1)
        ext_t.setStyle(std_table_style())
        story.append(ext_t)
    else:
        story.append(P("No triples extracted.", cell_muted))
    story.append(Spacer(1, 20))

    # ── CANONICALIZED TRIPLES ──
    story.append(section_heading("Canonicalized Triples"))
    story.append(Spacer(1, 5))
    story.append(P("Predicates normalized against the database schema. The Confidence column reflects alignment quality.", note_style))
    story.append(Spacer(1, 6))

    if canonicalized_triples:
        can_data = [[P(h, hdr_style) for h in ["Subject", "Original Predicate", "Object", "Canonical Predicate", "Conf."]]]
        for item in canonicalized_triples:
            orig  = item.get("original",  {})
            canon = item.get("canonical", {})
            conf  = item.get("match_confidence", "LOW")
            can_data.append([
                P(to_str(orig.get("subject",    "")), cell_style),
                P(to_str(orig.get("predicate",  "")), cell_muted),
                P(to_str(orig.get("object",     "")), cell_style),
                P(to_str(canon.get("predicate", "")), cell_style),
                conf_chip(conf),
            ])
        can_t = Table(can_data, colWidths=[W*0.14, W*0.24, W*0.15, W*0.35, W*0.12], repeatRows=1)
        can_t.setStyle(std_table_style())
        story.append(can_t)
    else:
        story.append(P("No canonicalized triples.", cell_muted))
    story.append(PageBreak())

    # ── LOGICAL INCONSISTENCIES ──
    story.append(section_heading("Logical Inconsistencies"))
    story.append(Spacer(1, 5))
    story.append(P("Contradictions detected between triples within the LLM response, independent of any external knowledge source.", note_style))
    story.append(Spacer(1, 6))

    if logical_inconsistencies:
        TYPE_COLORS = {
            "DIRECT_CONFLICT":        ("#7F1D1D", "#FEF2F2", "#FCA5A5"),
            "MONOTONICITY_VIOLATION": ("#78350F", "#FFFBEB", "#FCD34D"),
            "TIER_ORDER_VIOLATION":   ("#4C1D95", "#F5F3FF", "#C4B5FD"),
        }
        inc_data = [[P(h, hdr_style) for h in ["Label", "Type", "Severity", "Explanation", "Triples Involved"]]]
        for inc in logical_inconsistencies:
            inc_label = inc.get("label", "logically_inconsistent")
            inc_type  = inc.get("type", "")
            triples_text = "\n".join(
                f"{to_str(t.get('subject',''))} -> {to_str(t.get('predicate',''))} -> {to_str(t.get('object',''))}"
                for t in inc.get("triples_involved", [])
            )
            lbl_fg,  lbl_bg,  lbl_bd  = LBL.get(inc_label, ("#374151", "#F9FAFB", "#D1D5DB"))
            type_fg, type_bg, type_bd = TYPE_COLORS.get(inc_type, ("#374151", "#F9FAFB", "#D1D5DB"))
            inc_data.append([
                chip(inc_label, lbl_fg,  lbl_bg,  lbl_bd,  size=5.5),
                chip(inc_type,  type_fg, type_bg, type_bd, size=5.5),
                sev_chip(inc.get("severity", "LOW")),
                P(inc.get("explanation", ""), cell_style),
                P(triples_text.strip(), cell_hint),
            ])
        inc_t = Table(inc_data, colWidths=[W*0.16, W*0.16, W*0.08, W*0.28, W*0.32], repeatRows=1)
        inc_t.setStyle(std_table_style())
        story.append(inc_t)
    else:
        story.append(P("No logical inconsistencies detected.", cell_muted))

    story.append(Spacer(1, 20))

    # ── FACTUALLY INCONSISTENT TRIPLES ──
    story.append(section_heading("Factually Inconsistent Triples"))
    story.append(Spacer(1, 5))
    story.append(P("All triples labelled factually_inconsistent from the combined verification agent. Part A shows Pass-1 symbolic constraint violations. Part B shows Pass-2 hallucination verdicts with full evidence.", note_style))
    story.append(Spacer(1, 10))

    story.append(P("PART A — Symbolic Constraint Violations  (Pass 1)", S("pa", fontSize=7, fontName="Helvetica-Bold", textColor=NAVY_500, leading=10)))
    story.append(Spacer(1, 5))

    if symbolic_violations:
        sym_data = [[P(h, hdr_style) for h in ["Label", "Inconsistency Type", "Subject", "Predicate", "Object", "Constraint ID", "Explanation"]]]
        for sv in symbolic_violations:
            t    = sv.get("triple", {})
            sym_data.append([
                label_chip(sv.get("label", "factually_inconsistent")),
                chip(sv.get("inconsistency_type", "RULE_VIOLATION"), "#7F1D1D", "#FEF2F2", "#FCA5A5", size=5.5),
                P(to_str(t.get("subject", "")), cell_style),
                P(to_str(t.get("predicate", "")),  cell_muted),
                P(to_str(t.get("object", "")),   cell_style),
                P(sv.get("constraint_id", ""),   cell_hint),
                P(sv.get("explanation", ""),  cell_style),
            ])
        sym_t = Table(sym_data, colWidths=[W*0.12, W*0.13, W*0.12, W*0.13, W*0.12, W*0.09, W*0.29], repeatRows=1)
        sym_t.setStyle(std_table_style())
        story.append(sym_t)
    else:
        story.append(P("No symbolic constraint violations detected.", cell_muted))

    story.append(Spacer(1, 14))

    story.append(P("PART B — Hallucination Detection Verdicts  (Pass 2)", S("pb", fontSize=7, fontName="Helvetica-Bold", textColor=NAVY_500, leading=10)))
    story.append(Spacer(1, 5))

    if verdicts:
        col_w = [W*0.10, W*0.10, W*0.09, W*0.11, W*0.13, W*0.12, W*0.06, W*0.06, W*0.10, W*0.13]
        tbl_data = [[P(h, hdr_style) for h in ["Subject", "Predicate", "Object", "Verdict", "Label", "Inconsistency Type", "Match", "Violated", "Conf.", "Explanation"]]]

        for v in verdicts:
            verdict     = v.get("verdict", "UNVERIFIABLE")
            violated    = v.get("violated_component", "") or ""
            evidence    = v.get("evidence", "")
            lbl_val     = v.get("label", "")
            itype_val   = v.get("inconsistency_type", "")

            fg_h = VRD.get(verdict, ("#374151", "#F9FAFB", "#D1D5DB"))[0]
            viol_para = Paragraph(f'<font color="{fg_h}"><b>{violated}</b></font>' if violated else "—", S("vc2", fontSize=6, leading=9, alignment=TA_CENTER, textColor=INK_400))
            # After
            expl_para = P(trunc(v.get("explanation", "")), cell_style)
            expl_cell = [expl_para, Spacer(1, 2), P(trunc(f"Ev: {evidence}", 150), cell_hint)] if evidence else expl_para

            lbl_fg, lbl_bg, lbl_bd = LBL.get(lbl_val, ("#374151", "#F9FAFB", "#D1D5DB"))
            itype_chip = chip(itype_val, "#7F1D1D", "#FEF2F2", "#FCA5A5", size=5.5) if itype_val else P("—", cell_hint)

            tbl_data.append([
                P(v.get("subject",   ""), cell_style),
                P(v.get("predicate", ""), cell_muted),
                P(v.get("object",    ""), cell_style),
                verdict_chip(verdict),
                chip(lbl_val, lbl_fg, lbl_bg, lbl_bd, size=5.5),
                itype_chip,
                P(v.get("match_type", "NONE"), cell_hint),
                viol_para,
                P(conf_bar(v.get("confidence", 0.0)), cell_mono),
                expl_cell,
            ])

        hall_t = Table(tbl_data, colWidths=col_w, repeatRows=1)
        hall_t.setStyle(std_table_style())
        story.append(hall_t)
    else:
        story.append(P("No hallucination verdicts produced.", cell_muted))

    story.append(Spacer(1, 20))
    story.append(PageBreak())

    

    # ── VERDICT LEGEND ──
    story.append(HRFlowable(width="100%", thickness=0.4, color=RULE, spaceAfter=12))
    story.append(section_heading("Verdict Legend"))
    story.append(Spacer(1, 6))

    leg_data = [[P(h, hdr_style) for h in ["Verdict", "Description"]]]
    for vrd in ALL_VERDICTS:
        fg_h, bg_h, bd_h = VRD.get(vrd, ("#374151", "#F9FAFB", "#D1D5DB"))
        leg_data.append([
            Paragraph(f'<font color="{fg_h}"><b>{VERDICT_LABELS.get(vrd, vrd)}</b></font>', S(f"lv_{vrd[:4]}", fontSize=6.5, fontName="Helvetica-Bold", textColor=hex_color(fg_h), leading=10)),
            P(LEGEND_DESC.get(vrd, ""), cell_style),
        ])

    leg_t = Table(leg_data, colWidths=[W*0.26, W*0.74])
    leg_style = std_table_style()
    for i, vrd in enumerate(ALL_VERDICTS, 1):
        _, bg_h, _ = VRD.get(vrd, ("#374151", "#F9FAFB", "#D1D5DB"))
        leg_style.add("BACKGROUND", (0, i), (0, i), hex_color(bg_h))
    leg_t.setStyle(leg_style)
    story.append(leg_t)
    story.append(Spacer(1, 20))

    # ── Build ─────────────────────────────────────────────────────
    doc.build(story, onFirstPage=on_page, onLaterPages=on_page)
    print(f"[generate_report_pdf] Report saved to: {output_path}")
    state["hallucination_report_pdf"] = output_path
    return state

In [50]:
def run_pipeline(initial_state: dict) -> dict:
    state = dict(initial_state)

    pipeline_steps = [
        extract_triples,                  
        canonicalize_triples,
        detect_logical_inconsistencies,
        fetch_neo4j_evidence,
        combined_verification_agent,
        generate_report_pdf,
    ]

    for step_fn in pipeline_steps:
        print(f"\n{'='*60}")
        print(f"  Running step: {step_fn.__name__}")
        print(f"{'='*60}")
        state = step_fn(state)

    return state


# ─────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────
initial_state = {
    "user_input": """
 Context :
The following dataset describes the complete credit score spectrum used by financial institutions across the United States to evaluate borrower risk and determine lending conditions. Credit scores are numerical representations of a person's creditworthiness, calculated based on their history of paying bills, the amount of debt they currently carry, how long they have had credit accounts open, the mix of different types of credit they use, and how recently they have applied for new credit. The score itself is produced by credit bureaus such as Equifax, Experian, and TransUnion using proprietary models, the most widely used of which is the FICO score developed by the Fair Isaac Corporation.
The scale runs from a minimum of 300 to a maximum of 850. A score of 300 represents the most financially distressed and highest-risk borrowers, people who have likely experienced multiple defaults, bankruptcies, repossessions, or long stretches of missed payments. A score of 850 represents the ideal borrower in the eyes of a lender, someone with a long and spotless credit history, low utilization of available credit, no missed payments, and a healthy mix of credit products. In practice, very few people sit at either extreme. Most of the adult population falls somewhere in the middle of the range, and the practical consequences of being a few points higher or lower can be dramatic in terms of what financial products a person can access and at what cost.
The dataset you are about to read covers every score from 300 to 850, moving five points at a time, giving a total of 111 distinct score bands. For each score band, eight financial parameters are provided. These parameters are not invented arbitrarily. They reflect the real logic that lenders use when deciding whether to approve an application, what interest rate to charge, how much money to offer, and under what repayment conditions. Understanding how each parameter behaves across the score range reveals the deep and compounding inequality built into the credit system, where a person who starts with a low score pays more, gets less, and has fewer options, which often makes it harder to improve their score, which in turn keeps them paying more and getting less.
The first parameter is the risk tier. Lenders do not treat every score individually. They group scores into broad tiers that determine which internal lending products and policies apply. The five tiers used in this dataset are deep subprime, which covers scores from 300 to 579, subprime from 580 to 619, near prime from 620 to 659, prime from 660 to 739, and super prime from 740 to 850. Crossing from one tier into another is not a smooth transition. It is a hard boundary. A borrower at 619 and a borrower at 620 may have nearly identical financial histories, but one is classified as subprime and the other as near prime, and lenders treat them very differently as a result.
Now I will provide the parameters of some credit scores.

A 300 credit score has a risk tier of deep subprime, an interest rate of 22.94%, approval odds of 8%, a maximum debt-to-income ratio of 28%, a loan term of 12 months, a down payment of 22.1%, a credit limit multiplier of 0.31x, a default probability of 38.5%, and the lender pool includes hard money lenders. has collateral type real_estate
A 305 credit score has a risk tier of deep subprime and an interest rate of 22.61%, has collateral type real_estate
A 310 credit score has a risk tier of deep subprime and an interest rate of 22.19%. has collateral type real_estate
A 315 credit score has a risk tier of deep subprime and an interest rate of 21.88%. has collateral type real_estate
A 320 credit score has a risk tier of deep subprime and an interest rate of 21.71%. has collateral type real_estate
A 325 credit score has a risk tier of deep subprime and an interest rate of 21.44%. has collateral type real_estate
A 330 credit score has a risk tier of deep subprime and an interest rate of 21.03%. has collateral type real_estate
A 335 credit score has a risk tier of deep subprime and an interest rate of 20.88%. has collateral type real_estate
A 340 credit score has a risk tier of deep subprime and an interest rate of 20.71%. has collateral type real_estate
A 345 credit score has a risk tier of deep subprime and an interest rate of 20.44%. has collateral type real_estate
credit scored from 300 to 345 with incremental of 5 points has same approval odds, maximum debt-to-income, lender pool 
A 350 credit score has a risk tier of deep subprime, an interest rate of 20.19%, approval odds of 13%, a maximum debt-to-income ratio of 29%, and the lender pool includes specialty finance companies. has collateral type vechile
A 355 credit score has a risk tier of deep subprime and an interest rate of 19.88%. has collateral type vechile
A 360 credit score has a risk tier of deep subprime and an interest rate of 19.53%. has collateral type vechile
A 365 credit score has a risk tier of deep subprime and an interest rate of 19.21%. has collateral type vechile
A 370 credit score has a risk tier of deep subprime and an interest rate of 19.71%. has collateral type vechile
A 375 credit score has a risk tier of deep subprime and an interest rate of 18.84%. has collateral type vechile
A 380 credit score has a risk tier of deep subprime and an interest rate of 18.63%. has collateral type vechile
A 385 credit score has a risk tier of deep subprime and an interest rate of 18.41%. has collateral type vechile
A 390 credit score has a risk tier of deep subprime and an interest rate of 18.33%. has collateral type vechile
A 395 credit score has a risk tier of deep subprime and an interest rate of 18.09%. has collateral type vechile
credit scored from 350 to 395 with incremental of 5 points has same approval odds, maximum debt-to-income, lender pool .
credit scored from 300 to 395 with incremental of 5 points has same loan term , credit limit multiplier, down payment and default probability.

A 400 credit score has a risk tier of deep subprime, an interest rate of 17.94%, approval odds of 17%, a maximum debt-to-income ratio of 31%, a loan term of 24 months, a down payment of 19.4%, a credit limit multiplier of 0.54x, a default probability of 28.7%, and the lender pool includes credit unions only.
A 405 credit score has a risk tier of deep subprime and an interest rate of 17.61%.
A 410 credit score has a risk tier of deep subprime and an interest rate of 17.52%.
A 415 credit score has a risk tier of deep subprime and an interest rate of 17.29%.
A 420 credit score has a risk tier of deep subprime and an interest rate of 17.11%.
A 425 credit score has a risk tier of deep subprime and an interest rate of 16.88%.
A 430 credit score has a risk tier of deep subprime and an interest rate of 16.54%.
A 435 credit score has a risk tier of deep subprime and an interest rate of 16.41%.
A 440 credit score has a risk tier of deep subprime and an interest rate of 16.77%.
A 445 credit score has a risk tier of deep subprime and an interest rate of 16.19%.
credit scored from 400 to 445 with incremental of 5 points has same approval odds, maximum debt-to-income, lender pool 
A 450 credit score has a risk tier of deep subprime, an interest rate of 15.88%, approval odds of 22%, a maximum debt-to-income ratio of 33%, and the lender pool includes predatory lenders.
A 455 credit score has a risk tier of deep subprime and an interest rate of 16.07%.
A 460 credit score has a risk tier of deep subprime and an interest rate of 15.71%.
A 465 credit score has a risk tier of deep subprime and an interest rate of 15.44%.
A 470 credit score has a risk tier of deep subprime and an interest rate of 15.38%.
A 475 credit score has a risk tier of deep subprime and an interest rate of 15.19%.
A 480 credit score has a risk tier of deep subprime and an interest rate of 15.03%.
A 485 credit score has a risk tier of deep subprime and an interest rate of 14.81%.
A 490 credit score has a risk tier of deep subprime and an interest rate of 14.72%.
A 495 credit score has a risk tier of deep subprime and an interest rate of 14.54%.
credit scored from 450 to 495 with incremental of 5 points has same approval odds, maximum debt-to-income, lender pool 
credit scored from 400 to 495 with incremental of 5 points has same loan term , credit limit multiplier, down payment and default probability.

A 500 credit score has a risk tier of deep subprime, an interest rate of 14.41%, approval odds of 29%, a maximum debt-to-income ratio of 35%, a loan term of 12 to 24 months, a down payment of 15.4%, a credit limit multiplier of 1.06x, a default probability of 18.6%, and the lender pool includes specialty finance companies.
A 505 credit score has a risk tier of deep subprime and an interest rate of 14.19%.
A 510 credit score has a risk tier of deep subprime and an interest rate of 14.11%.
A 515 credit score has a risk tier of deep subprime and an interest rate of 13.94%.
A 520 credit score has a risk tier of deep subprime and an interest rate of 13.81%.
A 525 credit score has a risk tier of deep subprime and an interest rate of 13.54%.
A 530 credit score has a risk tier of deep subprime and an interest rate of 13.47%.
A 535 credit score has a risk tier of deep subprime and an interest rate of 13.29%.
credit scored from 500 to 535 with incremental of 5 points has same approval odds, maximum debt-to-income, lender pool 
A 540 credit score has a risk tier of deep subprime, an interest rate of 13.17%, approval odds of 36%, a maximum debt-to-income ratio of 37%, and the lender pool includes pawnshop credit providers.
A 545 credit score has a risk tier of deep subprime and an interest rate of 12.94%.
A 550 credit score has a risk tier of deep subprime and an interest rate of 12.81%.
A 555 credit score has a risk tier of deep subprime and an interest rate of 12.63%.
A 560 credit score has a risk tier of deep subprime and an interest rate of 12.54%.
A 565 credit score has a risk tier of deep subprime and an interest rate of 12.29%.
A 570 credit score has a risk tier of deep subprime and an interest rate of 12.21%.
A 575 credit score has a risk tier of deep subprime and an interest rate of 12.03%.
credit scores greater than or equal 300 and less than 580 has risk tier deep subprime 
credit scored from 540 to 575 with incremental of 5 points has same approval odds, maximum debt-to-income, lender pool 
credit scored from 500 to 575 with incremental of 5 points has same loan term , credit limit multiplier, down payment and default probability.
credit score from 300 to less than 580 has same risk tier 

A 580 credit score has a risk tier of subprime, an interest rate of 11.91%, approval odds of 41%, a maximum debt-to-income ratio of 39%, a loan term of 36 months, a down payment of 12.8%, a credit limit multiplier of 1.61x, a default probability of 11.3%, and the lender pool includes subprime banks.
A 585 credit score has a risk tier of subprime and an interest rate of 11.63%, and the lender pool includes subprime banks.
A 590 credit score has a risk tier of subprime and an interest rate of 11.56, and the lender pool includes subprime banks.
A 595 credit score has a risk tier of subprime and an interest rate of 11.38%, and the lender pool includes subprime banks.
A 600 credit score has a risk tier of subprime, an interest rate of 11.28%, and the lender pool includes online subprime lenders.
A 605 credit score has a risk tier of subprime and an interest rate of 11.03%, and the lender pool includes online subprime lenders.
A 610 credit score has a risk tier of subprime and an interest rate of 10.94%, and the lender pool includes online subprime lenders.
A 615 credit score has a risk tier of subprime, an interest rate of 10.77%, and the lender pool includes CDFI lenders.
credit scores greater than or equal 580 and less than 620 has risk tier subprime, and the lender pool includes CDFI lenders.
credit scored from 580 to 615 with incremental of 5 points has same approval odds, maximum debt-to-income. 
credit scored from 580 to 615 with incremental of 5 points has same loan term , credit limit multiplier, down payment  and default probability.
credit score from 580 to less than 620 has same risk tier

A 620 credit score has a risk tier of near prime, an interest rate of 10.67%, approval odds of 55%, a maximum debt-to-income ratio of 42%, a loan term of 36 to 48 months, a down payment of 10.6%, a credit limit multiplier of 2.07x, a default probability of 8.5%, and the lender pool includes community banks.
A 625 credit score has a risk tier of near prime and an interest rate of 10.41%.
A 630 credit score has a risk tier of near prime and an interest rate of 10.17%.
A 635 credit score has a risk tier of near prime and an interest rate of 10.13%.
credit scores from 620 to 635 with ncremental of 5 points have same lender pool.
A 640 credit score has a risk tier of near prime, an interest rate of 10.08%, and the lender pool includes online lenders.
A 645 credit score has a risk tier of near prime and an interest rate of 9.83%.
A 650 credit score has a risk tier of near prime and an interest rate of 9.74%.
A 655 credit score has a risk tier of near prime and an interest rate of 9.58%.
credit scores from 640 to 655 with ncremental of 5 points have same lender pool.
credit scores greater than or equal 620 and less than 660 has risk tier near prime. 
credit scored from 620 to 655 with incremental of 5 points has same approval odds, maximum debt-to-income.
credit scored from 620 to 655 with incremental of 5 points has same loan term , credit limit multiplier, down payment  and default probability.
credit score from 620 to less than 660 has same risk tier

A 660 credit score has a risk tier of prime, an interest rate of 9.49%, approval odds of 71%, a maximum debt-to-income ratio of 44%, a loan term of 48 to 60 months, a down payment of 8.3%, a credit limit multiplier of 2.63x, a default probability of 5.7%, and the lender pool includes regional banks.
A 665 credit score has a risk tier of prime and an interest rate of 9.24%.
A 670 credit score has a risk tier of prime and an interest rate of 9.18%.
A 675 credit score has a risk tier of prime and an interest rate of 9.14%.
A 680 credit score has a risk tier of prime and an interest rate of 8.89%.
A 685 credit score has a risk tier of prime and an interest rate of 8.64%.
A 690 credit score has a risk tier of prime and an interest rate of 8.561%.
A 695 credit score has a risk tier of prime and an interest rate of 8.43%.
credit scored from 660 to 695 with incremental of 5 points has same approval odds, maximum debt-to-income, lender pool 
credit scored from 660 to 695 with incremental of 5 points has same loan term , credit limit multiplier, down payment  and default probability.

A 700 credit score has a risk tier of prime, an interest rate of 8.31%, approval odds of 81%, a maximum debt-to-income ratio of 47%, a loan term of 60 months, a down payment of 6.3%, a credit limit multiplier of 3.19x, a default probability of 3.3%, and the lender pool includes most major banks.
A 705 credit score has a risk tier of prime and an interest rate of 8.08%.
A 710 credit score has a risk tier of prime and an interest rate of 7.97%.
A 715 credit score has a risk tier of prime and an interest rate of 7.84%.
credit scores from 700 to 715 with ncremental of 5 points have same lender pool.
A 720 credit score has a risk tier of prime, an interest rate of 7.74%, and the lender pool includes retail banks.
A 725 credit score has a risk tier of prime and an interest rate of 7.51%.
A 730 credit score has a risk tier of prime and an interest rate of 7.41%.
A 735 credit score has a risk tier of prime and an interest rate of 7.29%.
credit scores from 720 to 735 with ncremental of 5 points have same lender pool.
credit scores greater than or equal 660 and less than 740 has risk tier prime 
credit scored from 700 to 735 with incremental of 5 points has same approval odds, maximum debt-to-income. 
credit scored from 700 to 735 with incremental of 5 points has same loan term , credit limit multiplier, down payment  and default probability.
credit score from 660 to less than 740 has same risk tier

A 740 credit score has a risk tier of super prime, an interest rate of 7.19%, approval odds of 89%, a maximum debt-to-income ratio of 49%, a loan term of 60 to 84 months, a down payment of 4.3%, a credit limit multiplier of 3.99x, a default probability of 1.8%, and the lender pool includes all major banks.
A 745 credit score has a risk tier of super prime and an interest rate of 6.97%.
A 750 credit score has a risk tier of super prime and an interest rate of 6.86%.
A 755 credit score has a risk tier of super prime and an interest rate of 6.74%.
A 760 credit score has a risk tier of super prime and an interest rate of 6.64%.
A 765 credit score has a risk tier of super prime and an interest rate of 6.43%.
A 770 credit score has a risk tier of super prime and an interest rate of 6.33%.
A 775 credit score has a risk tier of super prime and an interest rate of 6.21%.
credit scored from 740 to 775 with incremental of 5 points has same approval odds, maximum debt-to-income, lender pool. 
A 780 credit score has a risk tier of super prime, an interest rate of 6.11%, approval odds of 95%, a maximum debt-to-income ratio of 51%, and the lender pool includes premium card issuers.
A 785 credit score has a risk tier of super prime and an interest rate of 5.91%.
A 790 credit score has a risk tier of super prime and an interest rate of 5.83%.
A 795 credit score has a risk tier of super prime and an interest rate of 5.71%. 
credit scored from 740 to 795 with incremental of 5 points has same loan term , credit limit multiplier, down payment  and default probability.

A 800 credit score has a risk tier of super prime, an interest rate of 5.63%, a loan term of 84 months, a down payment of 2.8%, a credit limit multiplier of 5.13x, and a default probability of 0.7%.
A 805 credit score has a risk tier of super prime and an interest rate of 5.43%.
A 810 credit score has a risk tier of super prime and an interest rate of 5.36%.
A 815 credit score has a risk tier of super prime and an interest rate of 5.24%.
credit scored from 780 to 815 with incremental of 5 points has same approval odds, maximum debt-to-income, lender pool 
A 820 credit score has a risk tier of super prime, an interest rate of 5.17%, approval odds of 97%, and the lender pool includes wealth management lenders.
A 825 credit score has a risk tier of super prime and an interest rate of 4.98%.
A 830 credit score has a risk tier of super prime and an interest rate of 4.91%.
A 835 credit score has a risk tier of super prime and an interest rate of 4.79%.
A 840 credit score has a risk tier of super prime and an interest rate of 4.72%.
A 845 credit score has a risk tier of super prime and an interest rate of 4.54%.
credit scored from 820 to 845 with incremental of 5 points has same approval odds, maximum debt-to-income, lender pool 
credit scored from 800 to 845 with incremental of 5 points has same loan term , credit limit multiplier, down payment  and default probability.

A 850 credit score has a risk tier of super prime, an interest rate of 4.36%, approval odds of 99%, a maximum debt-to-income ratio of 52%, a loan term of 84 months, a down payment of 2.0%, a credit limit multiplier of 6.50x, a default probability of 0.4%, and the lender pool includes all online lenders.
credit scores greater than or equal 740 and less than or equal 850 has risk tier super prime .
Interest rate is between 0 and 30 percent inclusive.
Interest rate decrease as credit score increase 
Approval odds is between 0 and 100 percent inclusive.
Down payment is between 0 and 50 percent inclusive.
meximum dept to incum ration is between 28 percent and 52 percent inclusive.
Allowed loan term values : ["12", "24", "36", "48", "60", "84", "12 to 24", "36 to 48", "48 to 60", "60 to 84","84"].
Default probability is between 0 and 100 percent inclusive.
The credit scores provided is with difference with 5 credit points.  

 "question": "Find the default probability for credit score 847",
        "reasoning_steps": [
            "Step 1:Find default probability for credit score 845",
            "Step 2:Find default probability for credit score 850",
            "Step 3:Find default probability for credit score 847"
        ]
Return your answers in this structured format for every asked value:
{"Subject":"<credit score>","Predicate":"<attribute>","Object":"<your answer>"}

""",
    "raw_response":          "",
    "extracted_triples":     [],
    "canonicalized_triples": [],
    "canonicalized_triples_Before" : [],
    "neo4j_evidence":        [],
    "symbolic_violations":   [],
    "hallucination_report":  "",
    "logical_inconsistencies":  [],
}

result = run_pipeline(initial_state)
print("User input:", result["user_input"][:200], "...")
print("\nRaw response from LLM:", result["raw_response"])
print("\nExtracted triples (JSON):")
print(json.dumps(result["extracted_triples"], indent=2))
print("\nCanonicalized triples before (JSON):")
print(json.dumps(result["canonicalized_triples_Before"], indent=2))
print("\nCanonicalized triples (JSON):")
print(json.dumps(result["canonicalized_triples"], indent=2))
print("\nEvidence triples (JSON):")
print(json.dumps(result["neo4j_evidence"], indent=2))


  Running step: extract_triples
To find the default probability for a credit score of 847, we need to follow the steps outlined:

## Step 1: Find default probability for credit score 845
The default probability for a credit score of 845 is not directly provided, but we can look at the closest values. For a credit score of 845, the information is not directly given, but we see that for scores around this range, such as 820 and 850, the default probabilities are provided as part of a range. However, the exact default probability for 845 is not directly stated, but we know that credit scored from 820 to 845 with incremental of 5 points has the same loan term, credit limit multiplier, down payment, and default probability.

## Step 2: Find default probability for credit score 850
For a credit score of 850, the default probability is given as 0.4%.

## Step 3: Find default probability for credit score 847
Given that the default probability decreases as the credit score increases and that t